## MinFin

This notebook implements *MinFin*, an analytical model developed to identify energy sector financing gaps for climate transitions in low- and middle-income countries (LMICs). MinFin quantifies investment needs, estimates financing costs, and evaluates anticipated sectoral cashflows to highlight financial shortfalls and explore feasible strategies to bridge them. The model’s primary goal is to facilitate better collaboration between energy and finance ministries, ensuring that energy planning aligns with national financial capacities and macroeconomic considerations. This Python version is based on the original Excel MinFin, which can be found [here](https://zenodo.org/records/14844158).

**Before you run**
- Open Jupyter with the **repository root** as the working directory so `data/...` paths resolve.
- Install dependencies: `pip install -r requirements.txt` (includes **plotly** for interactive figures).
- Optional editable install: `pip install -e .`

**Input workbooks** (set `file_path` in the next section)
- **Legacy:** `data/MINFin Energy Example Input File.xlsm` — Definitions + wide infrastructure sheets.
- **Pure input:** `data/MINFin Python Input File.xlsx` — long **INVESTMENT PLAN**, **TECHNOLOGY REGISTER**, revenue sheets.

Layout details: [docs/input_workbook_mapping.md](docs/input_workbook_mapping.md). Auto-detection: `from MinFin.data_processor import detect_workbook_format, WORKBOOK_FORMAT_PURE_INPUT`.

**Workflow** (main `MinFin` imports by section)

| Section | Modules |
|---------|---------|
| Definitions | `definitions_io`, `load_excel_data` |
| Technology / Disag | `technology_sheet_io`, `disag_tables` |
| Infrastructure | `input_extractor`, `read_infrastructure_input` |
| Financing baseline | `financing_baseline_extractor`, `financing_baseline_stats` |
| Investment / funding | `investment_needs_extra`, `funding_allocation` |
| Tariffs / repayment | `offtaker_tariffs`, `repayment_extras` |
| Dashboard / plots | `high_level_dashboard` (`hd`), `plotting_notebook`, `save_figure` |
| Export | `output_export` |

**Outputs** (regenerated; not versioned by default)
- `minfin_output/figures/` — HTML/PNG figures via `save_figure`
- `minfin_output/minfin_calculated_outputs.xlsx`, `minfin_output/minfin_technology_raw_input.xlsx` 


## Reading the input data

Set `file_path` to your workbook (legacy `.xlsm` or pure-input `.xlsx`), then list sheet names and `START_YEAR`. Uncomment only the path you are using. 


In [109]:
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

START_YEAR = 2025

# Legacy example workbook:
# file_path = "data/MINFin Energy Example Input File.xlsm"
# Pure-input workbook (default):
file_path = r"data/MINFin Input File - Apr26.xlsx"

with pd.ExcelFile(file_path, engine="openpyxl") as xls:
    sheet_names = xls.sheet_names

sheet_names


['COVER',
 'MACROECONOMIC',
 'TECHNOLOGY REGISTER',
 'INVESTMENT PLAN',
 'EXISTING INFRASTRUCTURE',
 'NEW INFRASTRUCTURE',
 'PPA REVENUE',
 'WHOLESALE REVENUE',
 'OTHER REVENUE']

In [110]:
import sys
sys.modules.pop("MinFin", None)  # clean the cache of MinFin module, in case it is changed between runs.
from MinFin import load_excel_data
# MinFin is a Python module designed specifically for this MINFin model
# We import functions from it   
# df_param_constraints, df_financing_baseline, df_funding_baseline, df_scenarios, df_currencies, df_technologies, df_technologies_classification =  load_excel_data(file_path)


# df_technologies.head(5) # show the first 5 rows of the technologies sheet
result = load_excel_data(file_path)
locals().update(result)
# result

In [111]:
df_technologies.head(20)

,Name,Description,Technology,Classification,Sector
0,BESS_TECH,Battery,Battery,Generation Renewable,
1,PWRBIO,Biomass,Biomass,Generation Renewable,
2,PWRDIST,Distribution,Distribution,Distribution Infrastructure,
3,PWRGEO,Geothermal,Geothermal,Generation Renewable,
4,PWRHYD,Hydropower,Hydropower,Generation Renewable,
5,PWRNGS,Natural Gas,Gas,Generation Fossil-Fuel,
6,PWRPHS,Pumped Hydro with Storage,Hydropower,Generation Renewable,
7,PWRSOL,Solar PV,Solar PV,Generation Renewable,
8,PWRTRN,Transmission,Transmission,Transmission Infrastructure,
9,PWRWND,Onshore Wind,Wind,Generation Renewable,


## Definitions → Python objects

- Convert Definition-sheet tables into dataclasses: technologies, constraints, financing/funding baselines, scenarios, and currencies. 


In [112]:
from MinFin.definitions_io import (
    BaseDefinition,
    ParameterConstraint,
    FinancingBaseline,
    FundingBaseline,
    Scenario,
    Currency,
    Technology,
    load_definitions_from_dataframe,
    load_currencies_from_dataframe,
    load_technologies_from_dataframe,
)

constraints = load_definitions_from_dataframe(df_param_constraints, ParameterConstraint)
baselines = load_definitions_from_dataframe(df_financing_baseline, FinancingBaseline)
fundings = load_definitions_from_dataframe(df_funding_baseline, FundingBaseline)
scenarios = load_definitions_from_dataframe(df_scenarios, Scenario)
currencies = load_currencies_from_dataframe(df_currencies)
technologies = load_technologies_from_dataframe(df_technologies.dropna())

# Pure-input: include parent technologies in NEW INFRASTRUCTURE without OSeMOSYS codes
# (CSP / Coal / Energy Exports / Direct*) as synthetic Technology rows; missing capital/generation/OPEX use zeros in cashflow.
from MinFin.technology_sheet_io import use_pure_input_tech_extraction, new_infrastructure_technology_list

if use_pure_input_tech_extraction(file_path):
    _existing_parents = {t.technology for t in technologies}
    _classification_map = (
        df_technologies_classification.set_index("Technology")["Classification"].to_dict()
    )
    for _name in new_infrastructure_technology_list(file_path):
        if _name in _existing_parents:
            continue
        technologies.append(
            Technology(
                name=_name,
                technology=_name,
                description="",
                classification=_classification_map.get(_name, ""),
            )
        )


In [113]:
df_technologies_classification.head(5)

,Technology,Classification
1,Biomass,Generation Renewable
2,Geothermal,Generation Renewable
3,Solar PV,Generation Renewable
4,Hydropower,Generation Renewable
5,Nuclear,Generation Renewable


In [114]:
# df_param_constraints.head(5)
# df_investment_needs.head(5)
# df_financing_baseline.head(5)
# df_funding_baseline.head(7)
# df_scenarios.head(5)
# df_currencies.head(5)

### Consumer segments → `organized_offtaker`

- Clean **Consumer segments**, detect header rows, forward-fill **Category**, and build the offtaker table used for tariffs and purchases. 


In [115]:
# dropna does not mutate unless inplace=True or assign back
consumer_segments.replace(['', ' ', None], np.nan, inplace=True)
# consumer_segments.dropna(subset=["Type"],how='all', inplace=True)
consumer_segments


,Name,Currency,Type,Offtaker
0,Generation,NaN,NaN,NaN
1,Transmission,NaN,NaN,NaN
2,Distribution,NaN,NaN,NaN
3,Commercial,KES,Distribution,Direct
4,Industrial,KES,Distribution,Direct
5,Residential,KES,Distribution,Direct
6,Exports,NaN,NaN,NaN


In [116]:

target_categories = ["Generation", "Transmission", "Distribution", "Exports"]

# 1. Identify header rows
is_header = consumer_segments['Name'].isin(target_categories) & consumer_segments['Type'].isna()

# 2. Forward-fill category labels
consumer_segments['Category'] = consumer_segments['Name'].where(is_header).ffill()

# 3. Non-header rows with Name; keep Category, Name, Currency
# One row per segment

organized_offtaker = consumer_segments[~is_header & consumer_segments['Name'].notna()][['Category', 'Name',"Currency"]].reset_index(drop=True)

organized_offtaker

,Category,Name,Currency
0,Distribution,Commercial,KES
1,Distribution,Industrial,KES
2,Distribution,Residential,KES


In [117]:
df_technologies_classification.head(5)

,Technology,Classification
1,Biomass,Generation Renewable
2,Geothermal,Generation Renewable
3,Solar PV,Generation Renewable
4,Hydropower,Generation Renewable
5,Nuclear,Generation Renewable


In [118]:
df_param_constraints.head(5)
df_investment_needs.head(5)
# df_financing_baseline.head(5)
# df_funding_baseline.head(7)
# df_scenarios.head(5)
# df_currencies.head(5)

,Name,Description


### Technology Disag (S1) / pure-input equivalent

- **Legacy** (`.xlsm`): extract each technology block from **Technology Disag (S1)** using fixed start rows (`TECH_START_ROWS`) and field positions.
- **Pure-input** (`MINFin Python Input File.xlsx`): the same logical fields come from **PPA REVENUE**, **WHOLESALE REVENUE**, and **OTHER REVENUE** (`extract_tech_data_pure_input` in `MinFin.technology_sheet_io`).
- Merge **offtaker share** columns from the organized consumer table, then `convert_tech_data_to_dataframes` for year-indexed series. 


In [119]:
from MinFin.technology_sheet_io import (
    TECH_START_ROWS,
    DEFAULT_FIELD_RELATIVE_POSITIONS,
    generate_share_configs,
    extract_tech_data_relative,
    extract_tech_data_pure_input,
    use_pure_input_tech_extraction,
    new_infrastructure_technology_list,
    infer_technology_disag_start_rows,
)

tech_to_class_map = df_technologies_classification.set_index("Technology")["Classification"].to_dict()
field_relative_positions = DEFAULT_FIELD_RELATIVE_POSITIONS

# Legacy workbook: one wide "Technology Disag (S1)" grid. Pure-input workbook: PPA / WHOLESALE / OTHER long sheets.
_tech_excel_pure = use_pure_input_tech_extraction(file_path)
if _tech_excel_pure:
    tech_start_rows = {name: None for name in new_infrastructure_technology_list(file_path)}
    sheet_name = None
    df_full = None
else:
    with pd.ExcelFile(file_path, engine="openpyxl") as _xls:
        sheet_name = "Technology Disag (S1)" if "Technology Disag (S1)" in _xls.sheet_names else "Technology Disag"
    df_full = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
    _inferred_start_rows = infer_technology_disag_start_rows(df_full, TECH_START_ROWS.keys())
    tech_start_rows = {**TECH_START_ROWS, **_inferred_start_rows}

all_tech_data = {}
for tech_name, start_row in tech_start_rows.items():
    offtake_share_configs = generate_share_configs(
        tech_name, organized_offtaker, tech_to_class_map
    )
    tech_fields = field_relative_positions | offtake_share_configs
    if _tech_excel_pure:
        tech_data = extract_tech_data_pure_input(
            file_path,
            tech_name,
            tech_fields,
        )
    else:
        tech_data = extract_tech_data_relative(
            df_full,
            sheet_name,
            tech_name,
            start_row,
            tech_fields,
        )
    all_tech_data[tech_name] = tech_data

field_relative_positions
list(all_tech_data.keys())


['Biomass',
 'Geothermal',
 'Solar PV',
 'Hydropower',
 'Nuclear',
 'Wind',
 'Oil',
 'Gas',
 'Transmission',
 'Distribution',
 'Imports',
 'Battery']

In [120]:
from MinFin.technology_sheet_io import convert_tech_data_to_dataframes

years = list(range(2025, 2071))
tech_dataframes = convert_tech_data_to_dataframes(all_tech_data, years)

_preview_name = next(iter(tech_dataframes))
print(f"Preview tech: {_preview_name}")
tech_dataframes[_preview_name].head()


Preview tech: Biomass


,total_grant_amount,ppa_currency,ppa_contracted_generation,ppa_standard_offtaker_share,ppa_direct_offtaker_tariff,ppa_standard_tariff,ppa_contracted_capacity,ppa_capacity_fee,ppa_penalty_tariff,redispatch_compensation_price,corporate_tax_rate,receivables,liabilities
2025,0.0,KES,14.888889,1.0,0.0,15.666667,0.0,0.0,0.0,0.0,0.3,0.0,0.0
2026,0.0,KES,58.694444,1.0,0.0,15.666667,0.0,0.0,0.0,0.0,0.3,0.0,0.0
2027,0.0,KES,58.694444,1.0,0.0,15.666667,0.0,0.0,0.0,0.0,0.3,0.0,0.0
2028,0.0,KES,93.694444,1.0,0.0,15.666667,0.0,0.0,0.0,0.0,0.3,0.0,0.0
2029,0.0,KES,93.694444,1.0,0.0,15.666667,0.0,0.0,0.0,0.0,0.3,0.0,0.0


### Per-technology annual tables

- `tech_dataframes` maps each technology to a year-indexed DataFrame (investment need, generation, tariffs, etc.). 


### Load the full Input workbook

- Read the **New Infrastructure (Input)** sheet once; following cells use `input_extractor('net_zero')` and named `load` / `load_block_for` calls. 


In [121]:
pd.set_option("future.no_silent_downcasting", True)  # avoid downcasting warning (truncated in original)

from MinFin.data_processor import read_infrastructure_input, detect_workbook_format, WORKBOOK_FORMAT_PURE_INPUT

# Legacy .xlsm: wide "New Infrastructure (Input)" grid. Pure-input .xlsx: this sheet does not exist — None here;
# use input_extractor(..., WORKBOOK_FORMAT_PURE_INPUT, file_path) in the next cells.
df_input_full = read_infrastructure_input(file_path)
_IS_PURE_INPUT_WB = detect_workbook_format(file_path) == WORKBOOK_FORMAT_PURE_INPUT

if df_input_full is not None:
    df_input_full.head(5)
else:
    print("Pure-input workbook: No New Infrastructure (Input)， INVESTMENT PLAN")


Pure-input workbook: No New Infrastructure (Input)， INVESTMENT PLAN


In [122]:
from MinFin import input_extractor
from MinFin.data_processor import WORKBOOK_FORMAT_PURE_INPUT

if _IS_PURE_INPUT_WB:
    net_zero_extrator = input_extractor("net_zero", WORKBOOK_FORMAT_PURE_INPUT, file_path)
    full_cap_cost_net_zero, df_ffe_net_zero, df_elec_production_nz = input_extractor.load_osemosys_input(
        "net_zero", None, WORKBOOK_FORMAT_PURE_INPUT, file_path
    )
else:
    net_zero_extrator = input_extractor("net_zero")
    full_cap_cost_net_zero, df_ffe_net_zero, df_elec_production_nz = input_extractor.load_osemosys_input(
        "net_zero", df_input_full
    )
net_zero_summary = net_zero_extrator.get_totals(df_input_full)

# Display sample results
net_zero_summary.head(3)
df_elec_production_nz.head()


Technology,BESS_TECH,PWRBIO,PWRDIST,PWRGEO,PWRHFO,PWRHYD,PWRHYD16,PWRIMP,PWRLFO,PWRNGS,PWRPHS,PWRSOL,PWRTRN,PWRURN,PWRWND
Year,,,,,,,,,,,,,,,
2025,0.4703,0.0536,50.1221,29.7337,2.4498,11.7784,0.3935,4.4506,0.0,0.0,0.0,1.5589,54.5992,0.0,7.0514
2026,0.5186,0.2113,51.6438,33.1873,0.4060,11.7784,0.7232,3.6673,0.0,0.0,0.0,1.8066,56.1896,0.0,7.6820
2027,0.5531,0.2113,54.5189,34.0681,0.9502,11.7784,0.8694,4.9733,0.0,0.0,0.0,1.8066,59.2532,0.0,8.1535
2028,0.5490,0.3373,57.6854,36.4531,1.0561,11.7784,0.9106,4.6700,0.0,0.0,0.0,1.8066,62.6198,0.0,9.3592
2029,0.5773,0.3373,61.2475,41.7923,0.8315,11.7784,1.0090,3.5221,0.0,0.0,0.0,1.8066,66.4146,0.0,9.3592


### Net Zero scenario (`input_extractor`)

- **Legacy:** read from **New Infrastructure (Input)** (wide sheet): capital cost, FFE, production, OPEX, potential generation, and scenario totals.
- **Pure-input:** the same blocks come from **INVESTMENT PLAN**; pass `workbook_format=WORKBOOK_FORMAT_PURE_INPUT` and `file_path` to `input_extractor` / `load_osemosys_input` (no `df_input_full`). 


In [123]:
from MinFin.data_processor import WORKBOOK_FORMAT_PURE_INPUT

if _IS_PURE_INPUT_WB:
    df_captial_cost = input_extractor.load_block_for(
        "net_zero", None, "capital_cost", WORKBOOK_FORMAT_PURE_INPUT, file_path
    )
else:
    df_captial_cost = input_extractor.load_block_for("net_zero", df_input_full, "capital_cost")
df_ffe = net_zero_extrator.load(df_input_full, "ffe")
df_captial_cost.head()


Technology,Year,BESS_TECH,PWRBIO,PWRDIST,PWRGEO,PWRHFO,PWRHYD,PWRHYD16,PWRIMP,PWRLFO,PWRNGS,PWRPHS,PWRSOL,PWRTRN,PWRURN,PWRWND
0,2025,103.2542,0.0,191.7485,278.2473,0.0,0.0,1.0453,0.0,0.0,0.0000,0.0,0.000,29.2420,0.0,19.0608
1,2026,0.0000,25.0,168.7515,544.2528,0.0,0.0,86.4096,0.0,0.0,0.0000,0.0,41.154,25.7349,0.0,86.8715
2,2027,0.0000,0.0,318.4243,403.9956,0.0,0.0,38.3269,0.0,0.0,83.0037,0.0,0.000,48.5603,0.0,74.4683
3,2028,0.0000,20.0,348.2791,331.8354,0.0,0.0,10.8012,0.0,0.0,19.8531,0.0,0.000,53.1132,0.0,170.5260
4,2029,0.0000,0.0,393.2781,1158.2439,0.0,0.0,25.7835,0.0,0.0,0.0000,0.0,0.000,59.9757,0.0,0.0000


In [124]:
df_ffe.head(5)

Technology,BESS_TECH,PWRBIO,PWRDIST,PWRGEO,PWRHFO,PWRHYD,PWRHYD16,PWRIMP,PWRLFO,PWRNGS,PWRPHS,PWRSOL,PWRTRN,PWRURN,PWRWND
Year,,,,,,,,,,,,,,,
2025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2026,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2027,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2028,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2029,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [125]:
df_elec_production = net_zero_extrator.load(df_input_full,'elec_production')
df_elec_production.head(5)

Technology,BESS_TECH,PWRBIO,PWRDIST,PWRGEO,PWRHFO,PWRHYD,PWRHYD16,PWRIMP,PWRLFO,PWRNGS,PWRPHS,PWRSOL,PWRTRN,PWRURN,PWRWND
Year,,,,,,,,,,,,,,,
2025,0.4703,0.0536,50.1221,29.7337,2.4498,11.7784,0.3935,4.4506,0.0,0.0,0.0,1.5589,54.5992,0.0,7.0514
2026,0.5186,0.2113,51.6438,33.1873,0.4060,11.7784,0.7232,3.6673,0.0,0.0,0.0,1.8066,56.1896,0.0,7.6820
2027,0.5531,0.2113,54.5189,34.0681,0.9502,11.7784,0.8694,4.9733,0.0,0.0,0.0,1.8066,59.2532,0.0,8.1535
2028,0.5490,0.3373,57.6854,36.4531,1.0561,11.7784,0.9106,4.6700,0.0,0.0,0.0,1.8066,62.6198,0.0,9.3592
2029,0.5773,0.3373,61.2475,41.7923,0.8315,11.7784,1.0090,3.5221,0.0,0.0,0.0,1.8066,66.4146,0.0,9.3592


In [126]:
df_opex = net_zero_extrator.load(df_input_full,'opex')
df_opex.head(5)

Technology,BESS_TECH,PWRBIO,PWRDIST,PWRGEO,PWRHFO,PWRHYD,PWRHYD16,PWRIMP,PWRLFO,PWRNGS,PWRPHS,PWRSOL,PWRTRN,PWRURN,PWRWND
Year,,,,,,,,,,,,,,,
2025,2.7890,0.2551,3.416041,148.9935,24.2603,23.1462,0.7648,80.3608,0.0,0.0000,0.0,5.6631,4.323751,0.0,32.5654
2026,2.7287,1.0052,3.519752,158.7308,16.9422,23.1462,1.4056,66.2176,0.0,0.0000,0.0,6.3908,4.449696,0.0,36.1738
2027,2.6683,1.0052,3.715703,156.7815,18.2704,23.1462,1.6897,89.7983,0.0,2.0239,0.0,6.2216,4.692305,0.0,38.8181
2028,2.6079,1.6053,3.931514,167.9166,18.5286,23.1462,1.7699,84.3224,0.0,2.5080,0.0,6.0582,4.958909,0.0,45.6968
2029,2.5476,1.6053,4.174286,196.0463,17.9808,23.1462,1.9610,63.5950,0.0,2.5080,0.0,5.9007,5.259422,0.0,45.5407


In [127]:
df_potential_generation_nz = net_zero_extrator.load(df_input_full,'potential_generation')
df_potential_generation_nz.head(5)

Technology,BESS_TECH,PWRBIO,PWRDIST,PWRGEO,PWRHFO,PWRHYD,PWRHYD16,PWRIMP,PWRLFO,PWRNGS,PWRPHS,PWRSOL,PWRTRN,PWRURN,PWRWND
Year,,,,,,,,,,,,,,,
2025,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2026,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2027,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2028,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2029,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## Least Cost Scenario 


In [128]:
from MinFin.data_processor import WORKBOOK_FORMAT_PURE_INPUT

if _IS_PURE_INPUT_WB:
    least_cost_extrator = input_extractor("least_cost", WORKBOOK_FORMAT_PURE_INPUT, file_path)
    full_cap_cost_least_cost, df_ffe_lc, df_elec_production_lc = input_extractor.load_osemosys_input(
        "least_cost", None, WORKBOOK_FORMAT_PURE_INPUT, file_path
    )
else:
    least_cost_extrator = input_extractor("least_cost")
    full_cap_cost_least_cost, df_ffe_lc, df_elec_production_lc = input_extractor.load_osemosys_input(
        "least_cost", df_input_full
    )
df_capital_cost_lc = least_cost_extrator.load(df_input_full, "capital_cost")
df_capital_cost_lc.head()



Technology,BESS_TECH,PWRBIO,PWRDIST,PWRGEO,PWRHFO,PWRHYD,PWRHYD16,PWRIMP,PWRLFO,PWRNGS,PWRPHS,PWRSOL,PWRTRN,PWRURN,PWRWND
Year,,,,,,,,,,,,,,,
2025,103.2542,0.0,191.7485,278.2473,0.0,0.0,1.0453,0.0,0.0,0.0000,0.0,0.000,29.2420,0.0,19.0608
2026,0.0000,25.0,168.7515,544.2528,0.0,0.0,86.4096,0.0,0.0,0.0000,0.0,41.154,25.7349,0.0,86.8715
2027,0.0000,0.0,318.4243,403.9956,0.0,0.0,38.3269,0.0,0.0,83.0037,0.0,0.000,48.5603,0.0,74.4683
2028,0.0000,20.0,348.2791,331.8354,0.0,0.0,10.8012,0.0,0.0,19.8531,0.0,0.000,53.1132,0.0,170.5260
2029,0.0000,0.0,393.2781,1158.2439,0.0,0.0,25.7835,0.0,0.0,0.0000,0.0,0.000,59.9757,0.0,0.0000


### Least Cost scenario — load blocks

- Load the same blocks as Net Zero (FFE, electricity production, OPEX, potential generation), then `get_totals` for the least-cost run. 


In [129]:
df_ffe_lc = least_cost_extrator.load(df_input_full,'ffe')
df_ffe_lc.head(3)


Technology,BESS_TECH,PWRBIO,PWRDIST,PWRGEO,PWRHFO,PWRHYD,PWRHYD16,PWRIMP,PWRLFO,PWRNGS,PWRPHS,PWRSOL,PWRTRN,PWRURN,PWRWND
Year,,,,,,,,,,,,,,,
2025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2026,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2027,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [130]:
df_elec_production_lc = least_cost_extrator.load(df_input_full,'elec_production')
df_elec_production_lc.head(3)


Technology,BESS_TECH,PWRBIO,PWRDIST,PWRGEO,PWRHFO,PWRHYD,PWRHYD16,PWRIMP,PWRLFO,PWRNGS,PWRPHS,PWRSOL,PWRTRN,PWRURN,PWRWND
Year,,,,,,,,,,,,,,,
2025,0.4703,0.0536,50.1221,29.7337,2.4498,11.7784,0.3935,4.4506,0.0,0.0,0.0,1.5589,54.5992,0.0,7.0514
2026,0.5186,0.2113,51.6438,33.1873,0.4060,11.7784,0.7232,3.6673,0.0,0.0,0.0,1.8066,56.1896,0.0,7.6820
2027,0.5531,0.2113,54.5189,34.0681,0.9502,11.7784,0.8694,4.9733,0.0,0.0,0.0,1.8066,59.2532,0.0,8.1535


In [131]:
df_opex_lc = least_cost_extrator.load(df_input_full,'opex')
df_opex_lc.head(3)


Technology,BESS_TECH,PWRBIO,PWRDIST,PWRGEO,PWRHFO,PWRHYD,PWRHYD16,PWRIMP,PWRLFO,PWRNGS,PWRPHS,PWRSOL,PWRTRN,PWRURN,PWRWND
Year,,,,,,,,,,,,,,,
2025,2.7890,0.2551,3.416041,148.9935,24.2603,23.1462,0.7648,80.3608,0.0,0.0000,0.0,5.6631,4.323751,0.0,32.5654
2026,2.7287,1.0052,3.519752,158.7308,16.9422,23.1462,1.4056,66.2176,0.0,0.0000,0.0,6.3908,4.449696,0.0,36.1738
2027,2.6683,1.0052,3.715703,156.7815,18.2704,23.1462,1.6897,89.7983,0.0,2.0239,0.0,6.2216,4.692305,0.0,38.8181


In [132]:
df_potential_generation_lc = least_cost_extrator.load(df_input_full,'potential_generation')
df_potential_generation_lc.head(3)

Technology,BESS_TECH,PWRBIO,PWRDIST,PWRGEO,PWRHFO,PWRHYD,PWRHYD16,PWRIMP,PWRLFO,PWRNGS,PWRPHS,PWRSOL,PWRTRN,PWRURN,PWRWND
Year,,,,,,,,,,,,,,,
2025,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2026,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2027,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [133]:
least_cost_summary =  least_cost_extrator.get_totals(df_input_full)
least_cost_summary.head(5)

,variable_cost,fixed_cost,co2_emission,carbon_price,carbon_credit_price,capital_cost,total_cost,annual_elec_production,cost_of_elec_in_pj,cost_of_elec,cost_of_co2
2025,326.537993,0.0,0.4391,0.0,0.0,622.5981,949.136093,162.6615,5.835038,1.620844,0.0
2026,320.710348,0.0,0.1041,0.0,0.0,978.1743,1298.884648,167.8141,7.740021,2.150006,0.0
2027,348.831208,0.0,0.1983,0.0,0.0,966.7791,1315.610308,177.1360,7.427120,2.063089,0.0
2028,363.050322,0.0,0.2378,0.0,0.0,954.4080,1317.458322,187.2255,7.036746,1.954652,0.0
2029,370.265308,0.0,0.1984,0.0,0.0,1637.2812,2007.546508,198.6758,10.104635,2.806843,0.0


 - We then calculate the CO2 savings from least cost and net zero scenarios 


In [134]:
co2_savings = pd.DataFrame(index=net_zero_summary.index)
if _IS_PURE_INPUT_WB:
    from MinFin.data_processor import emission_savings_series_from_investment_plan

    co2_savings["emission_saving"] = emission_savings_series_from_investment_plan(
        file_path, net_zero_summary.index
    )
    _carbon_px = net_zero_summary["carbon_credit_price"]
else:
    co2_savings["emission_saving"] = (
        least_cost_summary["co2_emission"] - net_zero_summary["co2_emission"]
    )
    _carbon_px = least_cost_summary["carbon_credit_price"]
co2_savings["co2_savings"] = co2_savings["emission_saving"] * _carbon_px
co2_savings.head()

,emission_saving,co2_savings
2025,0.0,0.0
2026,0.0,0.0
2027,0.0,0.0
2028,0.0,0.0
2029,0.0,0.0


 - In the following cell, we calculate the total costs for different scenarios 


In [135]:
#2025 -2070 period total
cost_keys = ['capital_cost','variable_cost','fixed_cost']
data = []
for key in cost_keys:
    data.append({
        "Scenarios": key,  # Scenarios: net zero / least cost
        "NetZero": net_zero_summary[key].sum(),# Million USD
        "LeastCost": least_cost_summary[key].sum(),#Million USD
    })
# net_zero_summary['capital_cost']
period_total = pd.DataFrame(data).T
period_total.columns = period_total.iloc[0] 
period_total = period_total[1:]
period_total['total'] = period_total.iloc[0:,:].sum(axis=1)
#Below is the data shown in the input sheet W137:AA138
period_total

Scenarios,capital_cost,variable_cost,fixed_cost,total
NetZero,135953.3506,48671.665106,0.0,184625.015706
LeastCost,135953.3506,48671.665106,0.0,184625.015706


### Slide window average 
- here we compute the 5 year windown average for various data.
- These are shown in the Excel Input sheet W141:AG151 


In [136]:
five_year_intervals = list(range(2025, 2070, 5))  # Generates [2025, 2030, 2035, ..., 2065]
df_5yr_avg = pd.DataFrame({"Year": five_year_intervals})
# Compute 5-year averages for each interval
for key in cost_keys:
    df_5yr_avg[f"nz_{key}"] = net_zero_summary[key].groupby(pd.cut(net_zero_summary.index, 
                                                              bins=list(range(2025, 2071, 5)), 
                                                              labels=five_year_intervals,right=False),observed=False) \
                                             .mean().reset_index(drop=True)
    df_5yr_avg[f"lc_{key}"] = least_cost_summary[key].groupby(pd.cut(net_zero_summary.index, 
                                                              bins=list(range(2025, 2071, 5)), 
                                                              labels=five_year_intervals,right=False),observed=False) \
                                             .mean().reset_index(drop=True)
    df_5yr_avg[f"delta_{key}"] = df_5yr_avg[f"lc_{key}"] - df_5yr_avg[f"nz_{key}"]
    
df_5yr_avg["emission_saving"] = co2_savings["co2_savings"].groupby(pd.cut(net_zero_summary.index, 
                                                              bins=list(range(2025, 2071, 5)), 
                                                              labels=five_year_intervals,right=False),observed=False).mean().reset_index(drop=True)    
df_5yr_avg.head(3)     

,Year,nz_capital_cost,lc_capital_cost,delta_capital_cost,nz_variable_cost,lc_variable_cost,delta_variable_cost,nz_fixed_cost,lc_fixed_cost,delta_fixed_cost,emission_saving
0,2025,1031.84814,1031.84814,0.0,345.879036,345.879036,0.0,0.0,0.0,0.0,0.0
1,2030,1350.73552,1350.73552,0.0,419.281524,419.281524,0.0,0.0,0.0,0.0,0.0
2,2035,1768.63014,1768.63014,0.0,542.363172,542.363172,0.0,0.0,0.0,0.0,0.0


In [137]:
net_zero_summary.head(5)

,variable_cost,fixed_cost,co2_emission,carbon_price,carbon_credit_price,capital_cost,total_cost,annual_elec_production,cost_of_elec_in_pj,cost_of_elec,cost_of_co2
2025,326.537993,0.0,0.4391,0.0,0.0,622.5981,949.136093,162.6615,5.835038,1.620844,0.0
2026,320.710348,0.0,0.1041,0.0,0.0,978.1743,1298.884648,167.8141,7.740021,2.150006,0.0
2027,348.831208,0.0,0.1983,0.0,0.0,966.7791,1315.610308,177.1360,7.427120,2.063089,0.0
2028,363.050322,0.0,0.2378,0.0,0.0,954.4080,1317.458322,187.2255,7.036746,1.954652,0.0
2029,370.265308,0.0,0.1984,0.0,0.0,1637.2812,2007.546508,198.6758,10.104635,2.806843,0.0


## Financing Baseline (historic instruments)

- Load historic financing, build **repayment statistics** 


### Here we extract the currency/exchange rate information: 
- We currenly generate random rates, can extract from the internet or set to be user-defined.
- It is not taking effect in the further processing of the data, as we use USD as default 


In [138]:
import numpy as np
import pandas as pd
from MinFin.data_processor import get_melted_currency_df
from MinFin.financing_baseline import financing_baseline_extractor, financing_baseline_stats

# Legacy: wide sheet «Financing Baseline». Pure input: «EXISTING INFRASTRUCTURE» + rates from «MACROECONOMIC».
fb = financing_baseline_extractor.from_workbook(file_path)
exchange_rates = fb.get_exchange_rates_by_year()
if len(exchange_rates) == 0:
    melted_currency = get_melted_currency_df()
else:
    melted_currency = exchange_rates.reset_index().melt(
        id_vars=["index"], var_name="Currency", value_name="Exchange Rate"
    )
melted_currency
exchange_rates.head() if len(exchange_rates) else melted_currency.head()


,USD,KES
2010,1.0,79.23
2011,1.0,88.81
2012,1.0,84.53
2013,1.0,86.12
2014,1.0,87.92


## Funding Baseline Sheet

Here we read and process the historic funding information from the excel sheet

``Requires user input`` 


In [139]:
# Load the "Funding Baseline" sheet from the Excel file

from MinFin import process_funding_baseline

# # Read the full "Funding Baseline" sheet
# df_funding_baseline_full = pd.read_excel(file_path, sheet_name="Funding Baseline", engine="openpyxl")
# df_final = process_funding_baseline(df_funding_baseline_full,melted_currency)
# df_funding_baseline=process_funding_baseline(df_funding_baseline_full,melted_currency)
# df_funding_baseline.iloc[50:]
# df_final.head(10)

 - And get the funding envolope from the input
  
> This is A5:R8 in the Funding Baseline sheet
>

*``Note that Budget statistics are to be determined in Excel, same calculation used here for data verification``*

`` Question to Team ``

> In LogReg mode, any negative values are ignores while others are not. 


In [140]:
# from MinFin import get_funding_envelope

# df_funding_envelope=get_funding_envelope(df_funding_baseline)
# # Average rows whose index looks like years; add as "Annual Average"
# df_funding_envelope.head()

## Investment Needs 
- with input we have processed, we now calculate the investment needs
> Investment Needs will be updated in a newer version, where names/categories are both from the definition sheet.
> FFRM related calculations removed 


In [141]:
from MinFin.investment_needs_extra import cal_invest_needs

(
    df_cap_by_class_lc,
    df_cap_by_tech_lc,
    df_cap_by_class_nz,
    df_cap_by_tech_nz,
    df_emission_savings,
    df_financing_needs_ffr,
    fossil_fuel_savings,
) = cal_invest_needs(
    technologies,
    least_cost_summary,
    net_zero_summary,
    df_ffe_lc,
    df_ffe,
    full_cap_cost_least_cost,
    full_cap_cost_net_zero,
    df_technologies,
    pure_input_file_path=file_path if _IS_PURE_INPUT_WB else None,
)


In [142]:
from MinFin.investment_needs_extra import get_category_sum

df_category_sum_lc = get_category_sum(df_cap_by_class_lc, df_cap_by_tech_lc)
df_category_sum_nz = get_category_sum(df_cap_by_class_nz, df_cap_by_tech_nz)
df_category_sum_nz.head(3)

Technology,Distribution Infrastructure,Generation Fossil-Fuel,Generation Renewable,Transmission Infrastructure,Battery,Biomass,Distribution,Gas,Geothermal,Hydropower,Imports,Nuclear,Oil,Solar PV,Transmission,Wind
Year,,,,,,,,,,,,,,,,
2025,191.7485,0.0000,401.6076,29.2420,103.2542,0.0,191.7485,0.0000,278.2473,1.0453,0.0,0.0,0.0,0.000,29.2420,19.0608
2026,168.7515,0.0000,783.6879,25.7349,0.0000,25.0,168.7515,0.0000,544.2528,86.4096,0.0,0.0,0.0,41.154,25.7349,86.8715
2027,318.4243,83.0037,516.7908,48.5603,0.0000,0.0,318.4243,83.0037,403.9956,38.3269,0.0,0.0,0.0,0.000,48.5603,74.4683


In [143]:
df_category_sum_incremental = df_category_sum_nz - df_category_sum_lc
df_category_sum_incremental.head(3)


Technology,Distribution Infrastructure,Generation Fossil-Fuel,Generation Renewable,Transmission Infrastructure,Battery,Biomass,Distribution,Gas,Geothermal,Hydropower,Imports,Nuclear,Oil,Solar PV,Transmission,Wind
Year,,,,,,,,,,,,,,,,
2025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2026,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2027,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [144]:
# df_category_sum_nz.head(3)

In [145]:
# df_emission_savings.T

In [146]:
# fossil_fuel_savings.T

In [147]:

# df_category_sum_incremental = df_category_sum_nz-df_category_sum_lc 
# df_category_sum_incremental.head(3)



In [148]:
# 

In [149]:
total_financing_needs_nz = df_category_sum_nz.sum(axis=1) / 2
total_financing_needs_lc = df_category_sum_lc.sum(axis=1) / 2
df_invest_need_summary = pd.DataFrame({
    "Net Zero": total_financing_needs_nz,
    "Least Cost": total_financing_needs_lc,
})
df_invest_need_summary.loc[:, "Total financing"] = df_invest_need_summary.loc[:, "Net Zero"]
df_invest_need_summary.head(3)


,Net Zero,Least Cost,Total financing
Year,,,
2025,622.5981,622.5981,622.5981
2026,978.1743,978.1743,978.1743
2027,966.7791,966.7791,966.7791


## Financing Baseline
``Requires user input``
Modalities of Financing – Grants, Debt and Equity


In the following cell, we import some functions from the MinFin module for Financning baseline to extract historic financing data and calculate repayment schedule and its statistics: 


In [150]:
import pandas as pd
from MinFin import financing_baseline_extractor, financing_baseline_stats
from MinFin.fx import macro_rates_dict_from_exchange_wide

fb = financing_baseline_extractor.from_workbook(file_path)
historical = fb.get_historical()
repayment_schedule = fb.cal_repayment_schedule(
    historical,
    convert_currency=True,
    rates_by_year=macro_rates_dict_from_exchange_wide(exchange_rates),
)
fbs = financing_baseline_stats(fb, repayment_schedule)
repayment_statistics = fbs.get_repayment_statistics()


In [151]:
repayment_schedule.head()

,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,...,2069,2070,Repayment,Name of Project,repay_years,Project ID,Sum of Repayment,Market Element,Grant Element,Average Annual Payment
0,0,0,0,0,8.892,7.70551,7.204624,6.827306,6.720178,6.427065,...,0,0,NaN,Geothermal | 2014 | 0,"[2014, 2015, 2016, 2017, 2018, 2019, 2020, 202...",0,98.811131,0.947912,0.052088,3.952445
1,0,0,0,0,0.081688,0.073151,0.070759,0.069452,0.070898,0.070419,...,0,0,NaN,Geothermal | 2014 | 1,"[2014, 2015, 2016, 2017, 2018, 2019, 2020, 202...",1,1.365356,NaN,NaN,0.054614
2,0,0,0,0,3.530438,3.1615,3.058089,3.001606,3.064127,3.043397,...,0,0,NaN,Geothermal | 2014 | 2,"[2014, 2015, 2016, 2017, 2018, 2019, 2020, 202...",2,82.937324,0.339693,0.660307,3.949396
3,0,0,0,0,2.8728,2.572587,2.488439,2.442477,2.493352,2.476484,...,0,0,NaN,Geothermal | 2014 | 3,"[2014, 2015, 2016, 2017, 2018, 2019, 2020, 202...",3,48.016841,NaN,NaN,1.920674
4,0,0,0,0,1.9152,1.715058,1.658959,1.628318,1.662235,1.650989,...,0,0,NaN,Geothermal | 2014 | 4,"[2014, 2015, 2016, 2017, 2018, 2019, 2020, 202...",4,32.011227,NaN,NaN,1.280449


In [152]:
# 

## Repaying the finance

- Repayments differ based on the Type of Finance and Repayment Schedule:

| Year | Annuity       |                  | Annuity + 3yr Grace |                  | EPP           |              | EPP + 3yr Grace on Principal Only |       | EPP + 3yr Grace on P & I |       | Lump Sum (Principal) |       | Lump Sum (P + Compound I) |       |
|------|----------------|------------------|----------------------|------------------|----------------|--------------|-----------------------------------|-------|----------------------------|-------|------------------------|-------|----------------------------|-------|
|      | Principal (P)  | Interest (I)     |  (P)        |  (I)     |  (P)  |  (I) |  (P)                     | (I)   |  (P)             | (I)   |  (P)          | (I)   |  (P)             | (I)   |
| 1    | ●●            | ●              |                      |                  | ●●            | ●●●          |                                  | ●●   |                            |    |                        | ●●   |                            |       |
| 2    | ●●            | ●              |                      |                  | ●●            | ●●          |                                  | ●●   |                            |    |                        | ●●   |                            |       |
| 3    | ●●            | ●              | ●●                 | ●              | ●●           | ●●          | ●●                              | ●●   | ●●                           | ●●   |                        | ●●   |                            |       |
| 4    | ●●            | ●              | ●●                  | ●              | ●●            | ●          | ●●                              | ●●   | ●●                        | ●●   |                        |●●       |                            |       |
| 5    | ●●            | ●              | ●●                  | ●             | ●●            | ●          | ●●                              | ●  | ●●                        | ●   | ●●●                    | ●●      | 
●●●                        | ●●●   |

In the following cell we show the repayment schedule, 


In [153]:
# historical

 >Discussion Point: Currency coversion 


In [154]:
# repayment_schedule.T.head(5)


In [155]:
# repayment_statistics

In [156]:
# EXISTING INFRASTRUCTURE has no Financial Institution column; skip this analysis
institution_shares = fbs.get_institution_shares()
institution_shares

,Share,Market Element
Bilateral Agency,0.0,NaN
Multilateral Agency,0.0,NaN
Foreign Government,0.0,NaN
National Government,0.0,NaN
Domestic Public Sector,0.0,NaN
Climate Funds,0.0,NaN
Commercial Bank,0.0,NaN
Private Equity Fund,0.0,NaN


In [157]:
# EXISTING INFRASTRUCTURE has no Financing Sector column; skip this analysis
financing_sector_shares = fbs.get_financing_sector_shares()
financing_sector_shares


,Share
Public,0.00000
Private,0.00000
Domestic,0.13103
International,0.86897


In [158]:
# Read Technology Disag (S1) sheet (legacy .xlsm only; pure-input file has no this sheet)
from MinFin.technology_sheet_io import use_pure_input_tech_extraction

if use_pure_input_tech_extraction(file_path):
    technology_disag_s1 = None
else:
    technology_disag_s1 = pd.read_excel(file_path, sheet_name="Technology Disag (S1)")
# technology_disag_s1.head()


### Technology Disag (S1) — structured extract

- Read the raw sheet, build **MultiIndex** columns (`Source`, `Category`, `Parameter`), and align technology row order with the workbook. 


In [159]:
# Build technology_disag_s1_data:
# - legacy .xlsm: read 'Technology Disag (S1)' wide block (Source × Category × Parameter)
# - pure input  : read 'NEW INFRASTRUCTURE' (same Source × Category × Parameter shape) via helper
from MinFin.technology_sheet_io import (
    use_pure_input_tech_extraction,
    technology_disag_s1_from_new_infrastructure,
)

if use_pure_input_tech_extraction(file_path):
    technology_disag_s1_raw = None
    technology_disag_s1_extracted = None
    technology_disag_s1_data = technology_disag_s1_from_new_infrastructure(file_path)
else:
    # Read raw (header=None) for multi-level headers
    technology_disag_s1_raw = pd.read_excel(
        file_path, sheet_name="Technology Disag (S1)", header=None
    )

    start_col = 3  # first financing parameter column
    end_col = 43  # through column AQ (slice end exclusive)
    start_row = 3
    end_row = 61  # 3 header rows + data block
    tech_name_col = 1  # column B: technology names

    technology_disag_s1_extracted = technology_disag_s1_raw.iloc[
        start_row:end_row, start_col:end_col
    ].copy()

    # First 3 rows become MultiIndex header
    header_level1 = technology_disag_s1_extracted.iloc[0, :].values
    header_level2 = technology_disag_s1_extracted.iloc[1, :].values
    header_level3 = technology_disag_s1_extracted.iloc[2, :].values

    # Fill NaN in headers with ffill / ''
    header_level1 = pd.Series(header_level1).ffill().fillna("").values
    header_level2 = pd.Series(header_level2).ffill().fillna("").values
    header_level3 = pd.Series(header_level3).fillna("").values  # last level: no ffill

    # Build MultiIndex columns
    technology_disag_s1_extracted.columns = pd.MultiIndex.from_arrays(
        [header_level1, header_level2, header_level3],
        names=["Source", "Category", "Parameter"],
    )

    technology_disag_s1_data = technology_disag_s1_extracted.iloc[3:].copy()
    # Index by technology name from column B (same rows as the data block) — never leave a
    # RangeIndex for later positional relabeling.
    _tech_names = (
        technology_disag_s1_raw.iloc[start_row + 3 : end_row, tech_name_col]
        .astype(str)
        .str.strip()
    )
    technology_disag_s1_data.index = _tech_names.values
    technology_disag_s1_data = technology_disag_s1_data[
        ~technology_disag_s1_data.index.isin(["", "nan", "None", "Technology"])
    ]
    technology_disag_s1_data = technology_disag_s1_data.apply(pd.to_numeric, errors="coerce")


In [160]:
technology_disag_s1_data

Source           Comm_Intl                                                     \
Category              Debt                                Equity                
Parameter    Interest Rate Grace Period Loan Term Rate of Return Project Life   
Biomass           0.180044            0     15.77       0.218244           30   
Geothermal        0.175744            0     15.77       0.213944           25   
Solar PV          0.176144            0     15.77       0.214344           24   
Hydropower        0.172744            0     15.77       0.200244           50   
Nuclear           0.176244            0     15.77       0.214444           60   
Wind              0.172744            0     15.77       0.202944           25   
Oil               0.000000            0      0.00       0.000000            0   
Gas               0.174144            0     15.77       0.212344           30   
Transmission      0.176244            0     15.77       0.214444           50   
Distribution      0.176244            0     15.77       0.214444           70   
Imports           0.000000            0      0.00       0.000000            0   
Battery           0.186644            0     15.77       0.224844           40   

Source                                                       \
Category     Financing Shares                                 
Parameter          Debt Share Equity Share Share of Finance   
Biomass                0.6302       0.3698         0.006079   
Geothermal             0.6302       0.3698         0.530562   
Solar PV               0.6302       0.3698         1.000000   
Hydropower             0.6302       0.3698         0.931010   
Nuclear                0.6302       0.3698         1.000000   
Wind                   0.6302       0.3698         0.476632   
Oil                    0.0000       0.0000         0.000000   
Gas                    0.6302       0.3698         0.389871   
Transmission           0.6302       0.3698         0.492531   
Distribution           0.6302       0.3698         0.559947   
Imports                0.0000       0.0000         0.000000   
Battery                0.6302       0.3698         0.044147   

Source                                       ...      Conc_DPS               \
Category     Foreign Currency Shares         ...          Debt                
Parameter                       Debt Equity  ... Interest Rate Grace Period   
Biomass                            1      1  ...        0.1269          7.1   
Geothermal                         1      1  ...        0.1269          7.1   
Solar PV                           1      1  ...        0.1269          7.1   
Hydropower                         1      1  ...        0.1269          7.1   
Nuclear                            1      1  ...        0.1269          7.1   
Wind                               1      1  ...        0.1269          7.1   
Oil                                0      0  ...        0.0000          0.0   
Gas                                1      1  ...        0.1269          7.1   
Transmission                       1      1  ...        0.1269          7.1   
Distribution                       1      1  ...        0.1269          7.1   
Imports                            0      0  ...        0.0000          0.0   
Battery                            1      1  ...        0.1269          7.1   

Source                                                               \
Category                       Equity              Financing Shares   
Parameter    Loan Term Rate of Return Project Life       Debt Share   
Biomass          22.64         0.1953           30             0.55   
Geothermal       22.64         0.1910           25             0.55   
Solar PV         22.64         0.1914           24             0.55   
Hydropower       22.64         0.1773           50             0.55   
Nuclear          22.64         0.1915           60             0.55   
Wind             22.64         0.1800           25             0.55   
Oil          

 > In the following cell, we match the order of the technologies as they appear in the excel sheet, without linking them to the cells, but linking them to previously defined variables. 


In [161]:

tech_order = {}
for idx, row in df_technologies.iterrows():
    tech_name = row["Technology"]
    if tech_name not in tech_order:
        tech_order[tech_name] = idx  # index of first occurrence

# Preferred display order: each class, then its technologies (no class-label rows in the data index)
ordered_techs = []
for class_name in df_technologies_classification["Classification"].unique():
    class_techs = [
        tech
        for tech in df_technologies_classification["Technology"]
        if tech_to_class_map.get(tech, "") == class_name
    ]
    class_techs_sorted = sorted(
        class_techs,
        key=lambda x: tech_order.get(x, float("inf")),
    )
    ordered_techs.extend(class_techs_sorted)

# Always align by technology *name*. Never assign ordered labels onto positional rows —
# that silently shifts parameters (e.g. Hydropower showing CSP Project Life).
_disag = technology_disag_s1_data.fillna(0)
if not isinstance(_disag.index, pd.RangeIndex) and len(_disag.index):
    _present = [name for name in ordered_techs if name in _disag.index]
    _missing = [name for name in _disag.index if name not in ordered_techs]
    technology_disag_s1_data = _disag.reindex(_present + _missing)
else:
    raise ValueError(
        "technology_disag_s1_data has no technology-name index; "
        "re-run the NEW INFRASTRUCTURE / Technology Disag load cell first."
    )

# Sanity: names must match financing rows (catches accidental positional relabeling)
_idx = pd.IndexSlice
if "Hydropower" in technology_disag_s1_data.index and "CSP" in _disag.index:
    _hp = float(
        technology_disag_s1_data.loc["Hydropower", _idx["Comm_Intl", "Equity", "Project Life"]]
    )
    _csp = float(_disag.loc["CSP", _idx["Comm_Intl", "Equity", "Project Life"]])
    if abs(_hp - _csp) < 1e-9 and abs(_hp - 50) < 1e-9:
        print(
            "WARNING: Hydropower Project Life equals CSP (50) — check NEW INFRASTRUCTURE "
            "or re-run disag load; Kenya Hydropower should be 24 if CSP is 50."
        )

technology_disag_s1_data.T.head(40)




Battery    Biomass  \
Source    Category                Parameter                                
Comm_Intl Debt                    Interest Rate      0.186644   0.180044   
                                  Grace Period       0.000000   0.000000   
                                  Loan Term         15.770000  15.770000   
          Equity                  Rate of Return     0.224844   0.218244   
                                  Project Life      40.000000  30.000000   
          Financing Shares        Debt Share         0.630200   0.630200   
                                  Equity Share       0.369800   0.369800   
                                  Share of Finance   0.044147   0.006079   
          Foreign Currency Shares Debt               1.000000   1.000000   
                                  Equity             1.000000   1.000000   
Comm_Dom  Debt                    Interest Rate      0.163700   0.157100   
                                  Grace Period       0.000000   0.000000   
                                  Loan Term         18.540000  18.540000   
          Equity                  Rate of Return     0.201900   0.195300   
                                  Project Life      40.000000  30.000000   
          Financing Shares        Debt Share         0.630200   0.630200   
                                  Equity Share       0.369800   0.369800   
                                  Share of Finance   0.187204   0.158984   
          Foreign Currency Shares Debt               0.000000   0.000000   
                                  Equity             0.000000   0.000000   
Conc_IFI  Debt                    Interest Rate      0.046900   0.046900   
                                  Grace Period       4.590000   4.590000   
                                  Loan Term         28.250000  28.250000   
          Equity                  Rate of Return     0.085100   0.085100   
                                  Project Life      40.000000  30.000000   
          Financing Shares        Debt Share         0.964138   0.964138   
                                  Equity Share       0.035862   0.035862   
                                  Share of Finance   0.352383   0.547437   
          Foreign Currency Shares Debt               1.000000   1.000000   
                                  Equity             1.000000   1.000000   
Conc_DPS  Debt                    Interest Rate      0.126900   0.126900   
                                  Grace Period       7.100000   7.100000   
                                  Loan Term         22.640000  22.640000   
          Equity                  Rate of Return     0.201900   0.195300   
                                  Project Life      40.000000  30.000000   
          Financing Shares        Debt Share         0.550000   0.550000   
                                  Equity Share       0.450000   0.450000   
                                  Share of Finance   0.416266   0.287500   
          Foreign Currency Shares Debt               0.000000   0.000000   
                                  Equity             0.000000   0.000000   

                                                    Geothermal  Hydropower  \
Source    Category                Parameter                                  
Comm_Intl Debt                    Interest Rate       0.175744    0.172744   
                                  Grace Period        0.000000    0.000000   
                                  Loan Term          15.770000   15.770000   
          Equity                  Rate of Return      0.213944    0.200244   
                                  Project Life       25.000000   50.000000   
          Financing Shares        Debt Share          0.630200    0.630200   
                                  Equity Share        0.369800    0.369800   
                                  Share of Finance    0.530562    0.931010   
          Foreign Currency Shares Debt                1.000000    1.000000   
   

In [162]:

# technology_disag_s1_data: index = technologies, columns = MultiIndex (Source, Category, Parameter)
# Example: column where Source=Comm_Intl, Category=Debt, Parameter=Interest Rate
# Concise pattern: IndexSlice + multiply/broadcast across sources
idx = pd.IndexSlice
fs = technology_disag_s1_data.loc[:, idx[:, 'Financing Shares', :]]
# Multiply pairwise, then sum by source/category to get per-tech totals
# Drop Parameter/Category so columns align on Source, then multiply
debt_share = fs.loc[:, idx[:, :, 'Debt Share']].droplevel(['Parameter', 'Category'], axis=1)
share_of_finance = fs.loc[:, idx[:, :, 'Share of Finance']].droplevel(['Parameter', 'Category'], axis=1)
weights_debt = debt_share * share_of_finance

equity_share = fs.loc[:, idx[:, :, 'Equity Share']].droplevel(['Parameter', 'Category'], axis=1)
weights_equity = equity_share * share_of_finance



In [163]:

technology_disag_s1_data.columns
# technology_disag_s1_data.columns is a pandas MultiIndex
# Inspect MultiIndex attributes
attrs = dir(technology_disag_s1_data.columns)

# Level names via .names
level_names = technology_disag_s1_data.columns.names

# Level-0 and level-1 labels
sources = technology_disag_s1_data.columns.get_level_values('Source').unique()
parameters = technology_disag_s1_data.columns.get_level_values('Parameter').unique()
categories = technology_disag_s1_data.columns.get_level_values('Category').unique()
# e.g. select one parameter such as "Interest Rate" across technologies


In [164]:
from MinFin.investment_needs_extra import cal_weighted_avg

weighted_averages = {}
for category in categories:
    cat = technology_disag_s1_data.loc[:, idx[:, category, :]]
    columns = cat.columns
    debt_mask = columns.get_level_values("Category") == category
    for parameter in columns[debt_mask].get_level_values("Parameter").unique():
        data = cat.loc[:, idx[:, :, parameter]].droplevel(["Parameter", "Category"], axis=1)
        if category in ["Debt"] or parameter in ["Debt"]:
            weighted_avg = cal_weighted_avg(data, weights_debt)
        elif category in ["Equity"] or parameter in ["Equity"]:
            weighted_avg = cal_weighted_avg(data, weights_equity)
        else:
            if parameter in ["Debt Share"]:
                weighted_avg = weights_debt.sum(axis=1)
            elif parameter in ["Equity Share"]:
                weighted_avg = weights_equity.sum(axis=1)
            else:
                weighted_avg = share_of_finance.sum(axis=1)

        weighted_averages[(category, parameter)] = weighted_avg

weighted_averages = pd.DataFrame(weighted_averages)

# Weighted Average Cost of Capital by technology
# (debt interest rate x debt share + equity rate of return x equity share)
weighted_averages[("Weighted Cost of Capital", "WACC")] = (
    weighted_averages[("Debt", "Interest Rate")] * weighted_averages[("Financing Shares", "Debt Share")]
    + weighted_averages[("Equity", "Rate of Return")] * weighted_averages[("Financing Shares", "Equity Share")]
)

# Explain "repeated" rows: when Share of Finance is concentrated on one source
# (e.g. Zambia all Conc_IFI), debt terms collapse to that source's values for every tech.
_sof_by_source = share_of_finance.mean()
_dominant = _sof_by_source.idxmax() if len(_sof_by_source) else None
if _dominant is not None and float(_sof_by_source.max()) > 0.9:
    print(
        f"Note: financing is concentrated on '{_dominant}' "
        f"(mean Share of Finance={float(_sof_by_source.max()):.2f}). "
        "Debt Interest Rate / Loan Term / WACC will look similar across technologies "
        "when that source's parameters are themselves similar in NEW INFRASTRUCTURE."
    )

weighted_averages


Debt                                 Equity  \
             Interest Rate Grace Period  Loan Term Rate of Return   
Battery           0.097262     4.457661  24.363106       0.198042   
Biomass           0.077536     4.488003  25.834974       0.185246   
Geothermal        0.131594     2.092270  20.357412       0.202897   
Hydropower        0.170794     0.131754  16.030208       0.198563   
Solar PV          0.176144     0.000000  15.770000       0.214344   
Wind              0.138877     1.964918  19.613839       0.190521   
Imports           0.000000     0.000000   0.000000       0.000000   
Nuclear           0.176244     0.000000  15.770000       0.214444   
Gas               0.084388     3.237697  24.573150       0.195577   
Oil               0.000000     0.000000   0.000000       0.000000   
Transmission      0.138061     2.052118  19.877814       0.202278   
Distribution      0.147689     1.931690  19.003500       0.203817   

                          Financing Shares                                \
             Project Life       Debt Share Equity Share Share of Finance   
Battery              40.0         0.714490     0.285510              1.0   
Biomass              30.0         0.789953     0.210047              1.0   
Geothermal           25.0         0.684623     0.315377              1.0   
Hydropower           50.0         0.628499     0.371501              1.0   
Solar PV             24.0         0.630200     0.369800              1.0   
Wind                 25.0         0.653289     0.346711              1.0   
Imports               0.0         0.000000     0.000000              0.0   
Nuclear              60.0         0.630200     0.369800              1.0   
Gas                  30.0         0.833946     0.166054              1.0   
Oil                   0.0         0.000000     0.000000              0.0   
Transmission         50.0         0.662269     0.337731              1.0   
Distribution         70.0         0.639792     0.360208              1.0   

             Foreign Currency Shares           Weighted Cost of Capital  
                                Debt    Equity                     WACC  
Battery                     0.514448  0.101442                 0.126036  
Biomass                     0.672997  0.104167                 0.100160  
Geothermal                  0.766316  0.644560                 0.154081  
Hydropower                  0.933529  0.926748                 0.181110  
Solar PV                    1.000000  1.000000                 0.190270  
Wind                        0.631582  0.520413                 0.156783  
Imports                     0.000000  0.000000                 0.000000  
Nuclear                     1.000000  1.000000                 0.190370  
Gas                         1.000000  1.000000                 0.102852  
Oil                         0.000000  0.000000                 0.000000  
Transmission                0.674226  0.554291                 0.159749  
Distribution                0.675614  0.583054                 0.167907

In [165]:
# If your MultiIndex is in .columns
columns = technology_disag_s1_data.columns  # or your MultiIndex

# Parameters where Category=='Debt'
# e.g. Index(['Interest Rate', 'Grace Period', 'Loan Term'], dtype='object')

In [166]:
technologies

[Technology(name='BESS_TECH', tech='Battery', description='Battery', classification='Generation Renewable'),
 Technology(name='PWRBIO', tech='Biomass', description='Biomass', classification='Generation Renewable'),
 Technology(name='PWRDIST', tech='Distribution', description='Distribution', classification='Distribution Infrastructure'),
 Technology(name='PWRGEO', tech='Geothermal', description='Geothermal', classification='Generation Renewable'),
 Technology(name='PWRHYD', tech='Hydropower', description='Hydropower', classification='Generation Renewable'),
 Technology(name='PWRNGS', tech='Gas', description='Natural Gas', classification='Generation Fossil-Fuel'),
 Technology(name='PWRPHS', tech='Hydropower', description='Pumped Hydro with Storage', classification='Generation Renewable'),
 Technology(name='PWRSOL', tech='Solar PV', description='Solar PV', classification='Generation Renewable'),
 Technology(name='PWRTRN', tech='Transmission', description='Transmission', classification='Tr

In [167]:
from MinFin.funding_allocation import InvestmentAllocator


### Funding allocation (`InvestmentAllocator`)

- Combine **Technology Disag (S1)** shares into per-source configs (`SourceConfig`) for each technology. 


In [168]:
from collections import defaultdict
import pandas as pd
from MinFin.funding_allocation import SourceConfig, TechnologyStats

tech_stats_dict = {}
tech_stats_by_class = defaultdict(list)

_disag_techs = set(technology_disag_s1_data.index)
sources = technology_disag_s1_data.columns.get_level_values(0).unique()

for tech in technologies:
    tech_stats = TechnologyStats.from_technology(tech)
    if tech.technology in df_category_sum_nz.columns:
        tech_stats.investment = df_category_sum_nz[tech.technology].to_dict()
    # Skip technologies in the registry but missing from NEW INFRASTRUCTURE financing params (e.g. Battery)
    if tech.technology in _disag_techs:
        for src in sources:
            tech_stats.financing_configs[src] = SourceConfig(
                raw_data=technology_disag_s1_data.loc[tech.technology, src]
            )

    tech_stats_dict[tech.technology] = tech_stats
    tech_stats_by_class[tech.classification].append(tech_stats)

# biomass_stats = tech_stats_dict["Biomass"]

# renewable_techs = tech_stats_by_class["Generation Renewable"]
# biomass_stats.financing_configs["Comm_Intl"].interest_rate


In [169]:
# Reset and reload financing config
for tech in technologies:
    tech_name = tech.technology
    t_stats = tech_stats_dict[tech_name]
    
    # Clear previous config
    t_stats.financing_configs = {}
    
    sources = technology_disag_s1_data.columns.get_level_values(0).unique()
    for src in sources:
        try:
            # Extract Series explicitly
            row_data = technology_disag_s1_data.loc[tech_name, src]
            
            # Wrap in config object explicitly
            config_obj = SourceConfig(raw_data=row_data)
            
            # Store in dict
            t_stats.financing_configs[src] = config_obj
            
        except KeyError:
            continue

# Verify last entry
print(f"Verification: last entry type is {type(t_stats.financing_configs[src])}")
print(technologies)

Verification: last entry type is <class 'MinFin.funding_allocation.SourceConfig'>
[Technology(name='BESS_TECH', tech='Battery', description='Battery', classification='Generation Renewable'), Technology(name='PWRBIO', tech='Biomass', description='Biomass', classification='Generation Renewable'), Technology(name='PWRDIST', tech='Distribution', description='Distribution', classification='Distribution Infrastructure'), Technology(name='PWRGEO', tech='Geothermal', description='Geothermal', classification='Generation Renewable'), Technology(name='PWRHYD', tech='Hydropower', description='Hydropower', classification='Generation Renewable'), Technology(name='PWRNGS', tech='Gas', description='Natural Gas', classification='Generation Fossil-Fuel'), Technology(name='PWRPHS', tech='Hydropower', description='Pumped Hydro with Storage', classification='Generation Renewable'), Technology(name='PWRSOL', tech='Solar PV', description='Solar PV', classification='Generation Renewable'), Technology(name='PW

 In the following cell, we form up a function to prepare the calculation of total generation. 


In [170]:
from MinFin.investment_needs_extra import aggregate_tech_production

df_generation_tech_sums = aggregate_tech_production(df_elec_production_nz, technologies)
df_potential_generation_tech_sums = aggregate_tech_production(df_potential_generation_nz, technologies)

# Technologies in NEW INFRASTRUCTURE without INVESTMENT PLAN OSeMOSYS rows (CSP / Coal / Energy Exports / Direct*)
# lack columns in aggregate tables; pad zero columns for downstream cashflow with zero investment/generation.
def _pad_zero_cols(df, names, year_index=None):
    import pandas as pd

    if df is None:
        return df
    if len(df.index) == 0:
        idx = year_index
        if idx is None:
            return df
        df = pd.DataFrame(index=pd.Index(idx, name=getattr(df.index, "name", None)))
    for n in names:
        if n not in df.columns:
            df[n] = 0.0
    return df

if _tech_excel_pure:
    _nia_full = list(all_tech_data.keys())
    _year_idx = None
    for _cand in (
        globals().get("years"),
        getattr(df_category_sum_nz, "index", None) if "df_category_sum_nz" in globals() else None,
    ):
        if _cand is not None and len(_cand):
            _year_idx = list(_cand)
            break
    df_generation_tech_sums = _pad_zero_cols(df_generation_tech_sums, _nia_full, _year_idx)
    df_potential_generation_tech_sums = _pad_zero_cols(
        df_potential_generation_tech_sums, _nia_full, _year_idx
    )
    df_category_sum_nz = _pad_zero_cols(df_category_sum_nz, _nia_full, _year_idx)
    df_category_sum_lc = _pad_zero_cols(df_category_sum_lc, _nia_full, _year_idx)
    df_cap_by_tech_nz = _pad_zero_cols(df_cap_by_tech_nz, _nia_full, _year_idx)
    df_cap_by_tech_lc = _pad_zero_cols(df_cap_by_tech_lc, _nia_full, _year_idx)

df_generation_tech_sums


,Battery,Biomass,Distribution,Geothermal,Oil,Hydropower,Imports,Gas,Solar PV,Transmission,Nuclear,Wind
Year,,,,,,,,,,,,
2025,0.4703,0.0536,50.1221,29.7337,2.4498,12.1719,4.4506,0.0000,1.5589,54.5992,0.0000,7.0514
2026,0.5186,0.2113,51.6438,33.1873,0.4060,12.5016,3.6673,0.0000,1.8066,56.1896,0.0000,7.6820
2027,0.5531,0.2113,54.5189,34.0681,0.9502,12.6478,4.9733,0.0000,1.8066,59.2532,0.0000,8.1535
2028,0.5490,0.3373,57.6854,36.4531,1.0561,12.6890,4.6700,0.0000,1.8066,62.6198,0.0000,9.3592
2029,0.5773,0.3373,61.2475,41.7923,0.8315,12.7874,3.5221,0.0000,1.8066,66.4146,0.0000,9.3592
2030,0.6300,1.0973,64.6927,42.9816,0.7442,13.2739,2.7614,0.0000,1.8066,70.0744,0.0000,12.0840
2031,1.1445,1.4125,68.0113,46.5130,0.7836,13.6475,2.5417,0.0000,1.8066,73.5814,0.0000,12.0840
2032,1.3177,1.5828,71.8601,47.6770,0.8580,13.8894,2.5472,0.0000,1.8066,77.6614,0.0000,14.9347
2033,1.3128,1.0770,75.5868,48.5712,0.7368,14.3175,2.5394,0.0000,1.8066,81.6008,0.0000,18.7934


 ```c
(XLOOKUP(C$63,'Financing Baseline'!$D$37:$BL$37,XLOOKUP('High Level Dashboard'!$B$5,'Financing Baseline'!$C$38:$C$47,'Financing Baseline'!$D$38:$BL$47))
/
XLOOKUP(C$63,'Financing Baseline'!$D$37:$BL$37,XLOOKUP('Technology Disag (S1)'!$C108,'Financing Baseline'!$C$38:$C$47,'Financing Baseline'!$D$38:$BL$47)))
```
*Reminder*: the above formula is to find the convert rate at a particular year (C63) between the currency used in *Highlevel Dashboard* and the PPA currency in *Tech Disag* 


In [171]:
from MinFin.offtaker_tariffs import (
    SEGMENT_MAP,
    UPSTREAM_CATEGORY_BY_NUM,
    _classify_segment,
    _get_tech_class,
    _get_offtaker_share_columns,
    _get_export_techs,
    _compute_export_gen,
    _slug,
    _empty_result,
    _upstream_totals,
    _weighted_components,
    _get_fx_to_dash,
    _get_offtaker_tariff_columns,
    _average_tariff,
    compute_generation_purchased,
    compute_capacity_purchased_and_avg_fee,
    compute_generation_purchased_series,
    compute_capacity_purchased_series,
)


### Offtaker-weighted generation purchases

- For Transmission, Distribution, and Exports technologies, populate **power purchased**, **tariff**, **capacity purchased**, and **capacity tariff** using `compute_generation_purchased_series` / `compute_capacity_purchased_series`. 


In [172]:

# 1. Instantiate allocator
alloc = InvestmentAllocator(technology_disag_s1_data)


In [173]:
# tech_offtakers={}
import numpy as np
from MinFin.index_alignment import apply_investment_to_tech_dataframes, assign_series_column, ensure_year_index
from MinFin.offtaker_tariffs import (
    calc_sale_price,
    compute_redispatch_compensation_mn_usd,
    compute_total_ppa_revenue,
    apply_network_receivables,
    flatten_multiindex_columns,
)
from MinFin.funding_allocation import TechnologyStats

apply_investment_to_tech_dataframes(tech_dataframes, df_category_sum_nz, grant_share=0.0)
df_opex_by_year = ensure_year_index(df_opex)

for tech_name, tech_data in all_tech_data.items():
    tech_dataframes[tech_name]["ppa_direct_offtaker_share"] = (
        1 - tech_dataframes[tech_name]["ppa_standard_offtaker_share"]
    )
    assign_series_column(
        tech_dataframes[tech_name],
        "total_generation",
        df_generation_tech_sums[tech_name] * 1000 / 3.6,
        context=f"{tech_name} total_generation",
    )
    _unit_hints = {
        k: v.get("unit", "")
        for k, v in tech_data.items()
        if isinstance(v, dict)
    }
    _redispatch_gwh = (
        df_potential_generation_tech_sums[tech_name] - tech_dataframes[tech_name]["total_generation"]
    )
    tech_dataframes[tech_name]["redispatch_compensation"] = compute_redispatch_compensation_mn_usd(
        tech_dataframes[tech_name], _redispatch_gwh, exchange_rates, unit_hints=_unit_hints
    )
    tech_dataframes[tech_name]["ppa_met_generation"] = np.minimum(
        tech_dataframes[tech_name]["total_generation"],
        tech_dataframes[tech_name]["ppa_contracted_generation"],
    )
    tech_dataframes[tech_name]["whole_sale_generation"] = (
        tech_dataframes[tech_name]["total_generation"] - tech_dataframes[tech_name]["ppa_met_generation"]
    )

    tech = TechnologyStats(name=tech_name, investment_needs=tech_dataframes[tech_name]["investment_need"])
    tech_dataframes[tech_name]["sale_price"] = calc_sale_price(tech_dataframes[tech_name], exchange_rates)
    tech_dataframes[tech_name]["total_wholesale_revnue"] = (
        tech_dataframes[tech_name]["sale_price"] * tech_dataframes[tech_name]["whole_sale_generation"].fillna(0)
    )
    tech_df = tech_dataframes[tech_name]

    tech_df["total_ppa_revenue"] = compute_total_ppa_revenue(
        tech_df, exchange_rates, unit_hints=_unit_hints
    )
    _years = tech.investment_needs.index.astype(int)
    lc_rates = exchange_rates[fb.currency].reindex(_years).astype(float)
    fc_rates = exchange_rates[fb.foreign_currency].reindex(_years).astype(float)

    tech_name_rows = df_technologies[df_technologies["Technology"] == tech_name]["Name"].values
    assign_series_column(
        tech_df,
        "opex",
        df_opex_by_year[tech_name_rows].sum(axis=1),
        context=f"{tech_name} opex",
    )

    df_result = tech.get_allocation_matrix(alloc, lc_rate=lc_rates, fc_rate=fc_rates)
    tech_stats_dict[tech_name].apply_allocation(df_result)

    df_result_flat = df_result.copy()
    df_result_flat = flatten_multiindex_columns(df_result_flat)

    tech_dataframes[tech_name] = pd.concat([tech_dataframes[tech_name], df_result_flat], axis=1)

    tech_stats_dict[tech_name].tech_df = tech_dataframes[tech_name].copy()

# biomass_df = tech_dataframes["Distribution"]
# biomass_df.T.head(60)
# biomass_df.T.head(50)


In [174]:
tech_name_row = df_technologies[df_technologies["Technology"] == tech_name]["Name"].values
tech_name_row

array(['BESS_TECH'], dtype=object)

In [175]:
# Only Transmission, Distribution, Energy Exports
dash_curr = fb.foreign_currency
for tech_name in tech_dataframes:
    tech_class_num = _classify_segment(_get_tech_class(tech_name, df_technologies))
    if tech_class_num <= 1:
        # Skip Generation
        tech_dataframes[tech_name]["power purchased"] = 0
        tech_dataframes[tech_name]["tariff"] = 0
        tech_dataframes[tech_name]["capacity purchased"] = 0
        tech_dataframes[tech_name]["capacity tariff"] = 0
        continue
    gp_series, tariff_series = compute_generation_purchased_series(
        tech_name,
        tech_dataframes, df_generation_tech_sums, df_technologies, organized_offtaker,
        exchange_rates=exchange_rates,
        dash_curr=dash_curr,
    )
    cap_series, avg_fee_series = compute_capacity_purchased_series(
        tech_name,
        tech_dataframes, df_technologies,
        exchange_rates=exchange_rates,
        dash_curr=dash_curr,
    )
    tech_dataframes[tech_name] = tech_dataframes[tech_name].assign(
        power_purchased=gp_series,
        tariff=tariff_series,
        capacity_purchased=cap_series,
        capacity_tariff=avg_fee_series,
        power_purchase_cost=lambda x: x["power_purchased"] * x["tariff"] + x["capacity_purchased"] * x["capacity_tariff"],
        )

apply_network_receivables(tech_dataframes, df_technologies)


In [176]:
# gp, tariff = compute_generation_purchased_series(
#     "Transmission", tech_dataframes, df_generation_tech_sums,
#     df_technologies, organized_offtaker, dash_curr=exchange_rates['USD'] 
# )
# tariff
# tech_dataframes["Transmission"].T

In [177]:
# 


In [178]:
# tech_dataframes["Distribution"].T
# 

In [179]:
gp_series, tariff_series = compute_generation_purchased_series(
    "Transmission", tech_dataframes, df_generation_tech_sums, df_technologies,
    organized_offtaker
)
cap_series, avg_fee_series = compute_capacity_purchased_series(
    "Transmission", tech_dataframes, df_technologies,
)

In [180]:
import numpy as np
import pandas as pd
from MinFin.repayment_extras import (
    _calc_debt_logic,
    _calc_equity_logic,
    cal_interest_cost,
    calculate_detailed_repayments,
)
from MinFin.output_export import compute_existing_financing_requirement_by_technology
from MinFin.technology_sheet_io import load_technology_alias_map

# Existing financing requirement = historic debt service from the financing baseline
# (repayment_schedule), attributed to each technology. The historic EXISTING INFRASTRUCTURE
# sheet labels rows by the register *description* (e.g. "Onshore Wind"), so map those to the
# parent model technology (e.g. "Wind") via the TECHNOLOGY REGISTER code table.
_tech_alias_map = load_technology_alias_map(file_path)
_existing_fin_by_tech = compute_existing_financing_requirement_by_technology(
    repayment_schedule,
    historical["Technology"],
    technology_map=_tech_alias_map,
)

# biomass_detail_df = calculate_detailed_repayments(
#     tech_stats_dict["Biomass"],
#     er=exchange_rates,
#     target_is_local=False,
#     local_curr_code="KES"
# )
financing_requirement_by_tech = {}
for tech_name, tech_stats_obj in tech_stats_dict.items():
    financing_requirement_by_tech[tech_name] = calculate_detailed_repayments(
        tech_stats_obj,
        er=exchange_rates,
        target_is_local=False,
        local_curr_code=fb.currency
    )
    financing_requirement_by_tech[tech_name].loc["Financing Requirement"] = (
        financing_requirement_by_tech[tech_name].sum(axis=0)
        + tech_dataframes[tech_name]["liabilities"].fillna(0)
    )
    # Existing Financing Requirement row (per technology), below the block like the workbook.
    _block_cols = financing_requirement_by_tech[tech_name].columns
    if _existing_fin_by_tech is not None and tech_name in _existing_fin_by_tech.index:
        _existing_row = _existing_fin_by_tech.loc[tech_name].reindex(_block_cols).fillna(0.0)
    else:
        _existing_row = pd.Series(0.0, index=_block_cols)
    financing_requirement_by_tech[tech_name].loc["Existing Financing Requirement"] = _existing_row
    financing_requirement_by_tech[tech_name].loc["total_financing_requirement"] = (
        pd.to_numeric(
            financing_requirement_by_tech[tech_name].loc["Financing Requirement"],
            errors="coerce",
        ).fillna(0.0)
        + pd.to_numeric(_existing_row, errors="coerce").fillna(0.0)
    )

# biomass_detail_df


In [181]:
# 

### Corporate tax and cashflow

- Compute **interest cost** from financing configs, then taxable profit and **corporate tax**, then **cashflow** (including grants, redispatch, and power purchase costs). 


In [182]:
#Update tech frames 
for tech_name, tech_stats_obj in tech_stats_dict.items():
    # Compact form, keep readable
    interest_cost = cal_interest_cost(
        tech_stats_obj,
        er=exchange_rates,
        target_is_local=False,
        local_curr_code=fb.currency
    )
    df = tech_dataframes[tech_name]
    df["interest_cost"]=interest_cost
    df["corporate_tax_expenses"] = (
        df["total_ppa_revenue"]
        + df["total_wholesale_revnue"]
        + (df["receivables"] if "receivables" in df.columns else 0)
        - df["opex"]
        - (df["power_purchase_cost"] if "power_purchase_cost" in df.columns else 0)
        - interest_cost
    ) * df["corporate_tax_rate"].fillna(0)
    df["corporate_tax_expenses"] = df["corporate_tax_expenses"].clip(lower=0)
    df["cashflow"] = (
        df["total_ppa_revenue"]
        + df["total_wholesale_revnue"]
        + df["receivables"]
        - df["opex"]
        - df["corporate_tax_expenses"]
        -(df["power_purchase_cost"] if "power_purchase_cost" in df.columns else 0)
        + df["redispatch_compensation"]
        + (df["total_grant_amount"]-df_cap_by_tech_nz[tech_name]).clip(lower=0)  
        )

In [183]:
# financing_requirement_by_tech['Biomass'].T.head(30)
# # tech_stats_dict['Solar PV'].financing_configs["Comm_Dom"]
# tech_dataframes['Solar PV']['interest_cost']

In [184]:
from MinFin.disag_tables import _class_to_segment, sum_tech_params_by_segment, _sum_tech_params

financing_summary = pd.concat(financing_requirement_by_tech.values()).groupby(level=0).sum()

financing_summary.loc["Financing Requirement", :] -= _sum_tech_params(tech_dataframes, "liabilities")
financing_summary.loc["Cashflows", :] = _sum_tech_params(tech_dataframes, "cashflow")
financing_summary = pd.concat(
    [
        financing_summary,
        sum_tech_params_by_segment(tech_dataframes, "total_generation", tech_to_class_map).T,
    ],
    axis=0,
)

financing_summary


,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,...,2061,2062,2063,2064,2065,2066,2067,2068,2069,2070
Equity (Comm_Dom),6.442860,14.934511,23.027867,32.079661,46.365071,59.269223,69.882019,82.012303,94.847153,104.205217,...,662.952847,690.841331,713.576766,740.422366,778.120786,799.025488,822.244092,841.809547,860.468776,885.362445
Equity (Comm_Intl),22.469291,65.989273,106.830810,145.761722,215.963779,270.333025,318.426023,369.087483,425.811869,497.060456,...,3459.441210,3605.528397,3726.306641,3873.368575,4036.554972,4159.892014,4292.207040,4413.852034,4538.573817,4678.794394
Equity (Conc_DPS),11.796173,24.007099,36.507816,50.428798,71.929481,91.106234,110.754549,128.789977,146.250260,160.513688,...,991.476176,1030.031382,1063.048797,1099.580533,1148.209431,1178.279535,1210.903162,1238.534508,1266.618270,1300.853565
Equity (Conc_IFI),0.346175,0.800091,1.325351,1.766564,2.588819,3.302850,3.996197,4.497606,4.951345,5.480570,...,33.410925,34.772004,35.918499,37.259810,38.769297,39.876889,41.131815,42.265525,43.489619,44.721183
Existing Financing Requirement,607.980038,610.130341,583.844079,557.321236,529.433916,508.077480,475.576029,432.782553,402.623598,372.119247,...,10.892318,6.537134,6.358488,4.700702,4.171392,4.061330,3.954172,3.849841,3.748264,3.649366
Financing Requirement,69.207469,190.174853,298.196567,401.252308,582.709921,732.459641,862.881645,993.118377,1134.709966,1308.743506,...,7189.066127,7463.178412,7699.270289,7970.976713,8329.695317,8562.071558,8795.084442,8996.067221,9189.205386,9467.835690
Loans (Comm_Dom),1.781870,3.951641,5.953666,8.356322,11.487952,15.190162,17.375710,20.686080,24.270770,25.357984,...,122.581625,127.642894,132.819386,136.585452,145.149781,148.811347,153.537549,156.809263,158.726482,165.803503
Loans (Comm_Intl),22.767355,73.716839,113.583051,148.209639,214.126106,265.107527,306.039467,344.507472,389.192685,460.517875,...,1636.756680,1683.840989,1728.719225,1777.935276,1864.794330,1909.261716,1941.145953,1961.668732,1973.482874,2034.314381
Loans (Conc_DPS),2.437129,4.183324,6.080241,8.313103,11.266682,14.613169,18.653913,22.787292,26.340606,28.964682,...,168.236786,172.671223,176.891000,179.731316,186.252093,189.944064,193.585723,198.105464,201.877231,207.250672
Loans (Conc_IFI),1.166615,2.592076,4.887764,6.336500,8.982031,13.537452,17.753765,20.750165,23.045278,26.643035,...,114.209879,117.850193,121.989974,126.093386,131.844627,136.980505,140.329108,143.022148,145.968317,150.735547


In [185]:

# tech_dataframes['Solar PV'].T

In [186]:
# tech_dataframes is a dict (no slice). To concat each tech's "total_grant_amount" by year/index, use e.g.:

# Concat each tech's total_grant_amount along columns (aligned on year/index)
total_grant_amount_df = pd.concat(
    [df["total_grant_amount"] for df in tech_dataframes.values() if "total_grant_amount" in df.columns],
    axis=1
)
total_investment_need_df = pd.concat(
    [df["investment_need"] for df in tech_dataframes.values() if "investment_need" in df.columns],
    axis=1
)
total_grant_amount_df.columns = list(tech_dataframes.keys())
total_investment_need_df.columns = list(tech_dataframes.keys())
total_investment_need_grant_covered = total_grant_amount_df.sum(axis=1)
total_investment_need_financing_need = total_investment_need_df.sum(axis=1)
# Stack into one DataFrame
df_investment_need_by_source = pd.DataFrame({
    'Grant Covered': total_investment_need_grant_covered,
    'Financing Need': total_investment_need_financing_need,
    'Net Zero': df_invest_need_summary['Net Zero']
}).T
df_investment_need_by_source

,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,...,2061,2062,2063,2064,2065,2066,2067,2068,2069,2070
Grant Covered,0.0000,2.0577,0.0000,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,6.245295,0.858485,0.99583,0.943005,1.043375,1.04866,1.04866,1.04866,1.04866,4.592455
Financing Need,622.5981,976.1166,966.7791,954.408,1637.2812,1393.0034,1262.9498,1269.0351,1355.3975,1473.2918,...,4717.481705,4456.056015,4569.53987,4663.290395,5840.963625,4906.73514,4715.95354,4543.93854,4436.66834,5448.561645
Net Zero,622.5981,978.1743,966.7791,954.408,1637.2812,1393.0034,1262.9498,1269.0351,1355.3975,1473.2918,...,4723.727000,4456.914500,4570.53570,4664.233400,5842.007000,4907.78380,4717.00220,4544.98720,4437.71700,5453.154100


In [187]:
organized_offtaker

,Category,Name,Currency
0,Distribution,Commercial,KES
1,Distribution,Industrial,KES
2,Distribution,Residential,KES


In [188]:

# 2. Instantiate technology stats
# tech = TechnologyStats(
#     name='Biomass', 
#     investment_needs=tech_dataframes['Biomass']["investment_need"]
# )

# 3. Build full allocation matrix
_years = tech.investment_needs.index.astype(int)
lc_rates = exchange_rates[fb.currency].reindex(_years).astype(float)
fc_rates = exchange_rates[fb.foreign_currency].reindex(_years).astype(float)
df_result = tech.get_allocation_matrix(alloc, lc_rate=lc_rates, fc_rate=fc_rates)
from MinFin.pure_input_blocks import read_cover_scenario, pure_input_scenario_label, export_scenario_label
cover_scenario = read_cover_scenario(file_path)
scenario_label = export_scenario_label(file_path)
scenario_label

'Net Zero'

In [189]:
# Export all calculated outputs before building High Level Dashboard
# The workbook is organized in broad sections similar to the legacy workbook tabs.
from pathlib import Path
import re
import pandas as pd
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from MinFin.excel_io import load_workbook_variable_units
import importlib
import MinFin.output_export as _minfin_output_export
importlib.reload(_minfin_output_export)
from MinFin.output_export import (
    export_technology_output_workbook,
    export_technology_parameter_raw_workbook,
)
variable_units = load_workbook_variable_units(file_path)

output_dir = Path("minfin_output")
output_dir.mkdir(parents=True, exist_ok=True)
output_excel_path = output_dir / "minfin_calculated_outputs.xlsx"
tech_raw_excel_path = output_dir / "minfin_technology_raw_input.xlsx"
tech_output_excel_path = output_dir / "minfin_technology_output_in_osemosys_way.xlsx"


def _safe_sheet_name(name: str, used: set[str]) -> str:
    base = re.sub(r"[\\/*?:\[\]]", "_", str(name)).strip() or "Sheet"
    base = base[:31]
    candidate = base
    i = 1
    while candidate in used:
        suffix = f"_{i}"
        candidate = f"{base[:31 - len(suffix)]}{suffix}"
        i += 1
    used.add(candidate)
    return candidate


def _to_frame(obj):
    if obj is None:
        return None
    if isinstance(obj, pd.Series):
        return obj.to_frame()
    if isinstance(obj, pd.DataFrame):
        return obj
    return None


def _write_df(writer, sheet_name: str, obj, index: bool = True):
    df = _to_frame(obj)
    if df is None or df.empty:
        return None
    final_sheet = _safe_sheet_name(sheet_name, used_sheet_names)
    df.to_excel(writer, sheet_name=final_sheet, index=index)
    export_index.append({"Sheet": final_sheet, "Object": sheet_name, "Rows": len(df), "Columns": len(df.columns)})
    return final_sheet


def _format_workbook(path: Path):
    from openpyxl import load_workbook

    wb = load_workbook(path)
    header_fill = PatternFill("solid", fgColor="D9EAF7")
    title_fill = PatternFill("solid", fgColor="1F4E78")
    title_font = Font(color="FFFFFF", bold=True)
    for ws in wb.worksheets:
        ws.freeze_panes = "B2"
        ws.sheet_view.showGridLines = False
        if ws.max_row >= 1:
            for cell in ws[1]:
                cell.font = Font(bold=True)
                cell.fill = header_fill
                cell.alignment = Alignment(horizontal="center")
        if ws.title in {"Export Index", "Run Metadata"}:
            for cell in ws[1]:
                cell.fill = title_fill
                cell.font = title_font
        ws.auto_filter.ref = ws.dimensions
        for col_idx in range(1, min(ws.max_column, 40) + 1):
            col_letter = get_column_letter(col_idx)
            max_len = 0
            for cell in ws[col_letter][: min(ws.max_row, 200)]:
                if cell.value is not None:
                    max_len = max(max_len, len(str(cell.value)))
            ws.column_dimensions[col_letter].width = min(max(max_len + 2, 10), 35)
    wb.save(path)


used_sheet_names = set()
export_index = []

# Main summary outputs
summary_objects = {
    "Net Zero Summary": globals().get("net_zero_summary"),
    "Least Cost Summary": globals().get("least_cost_summary"),
    "CO2 Savings": globals().get("co2_savings"),
    "Five Year Average": globals().get("df_5yr_avg"),
    "Investment Need Summary": globals().get("df_invest_need_summary"),
    "Financing Summary": globals().get("financing_summary"),
    "Repayment Statistics": globals().get("repayment_statistics"),
}

investment_objects = {
    "NZ Capital by Class": globals().get("df_cap_by_class_nz"),
    "NZ Capital by Tech": globals().get("df_cap_by_tech_nz"),
    "LC Capital by Class": globals().get("df_cap_by_class_lc"),
    "LC Capital by Tech": globals().get("df_cap_by_tech_lc"),
    "NZ Category Sum": globals().get("df_category_sum_nz"),
    "LC Category Sum": globals().get("df_category_sum_lc"),
    "Emission Savings": globals().get("df_emission_savings"),
    "Fossil Fuel Savings": globals().get("fossil_fuel_savings"),
    "FFR Financing Needs": globals().get("df_financing_needs_ffr"),
}

financing_objects = {
    "Historical Financing": globals().get("historical"),
    "Repayment Schedule": globals().get("repayment_schedule"),
    "Funding Baseline Processed": globals().get("df_funding_baseline"),
    "Funding Envelope": globals().get("df_funding_envelope"),
}

with pd.ExcelWriter(output_excel_path, engine="openpyxl") as writer:
    # Metadata / index sheets first
    metadata = pd.DataFrame(
        [
            {"Item": "Input workbook", "Value": str(globals().get("file_path", ""))},
            {"Item": "Output workbook", "Value": str(output_excel_path)},
            {"Item": "Export note", "Value": "Calculated outputs before High Level Dashboard"},
        ]
    )
    meta_sheet = _safe_sheet_name("Run Metadata", used_sheet_names)
    metadata.to_excel(writer, sheet_name=meta_sheet, index=False)

    for name, obj in summary_objects.items():
        _write_df(writer, f"Summary - {name}", obj)
    for name, obj in investment_objects.items():
        _write_df(writer, f"Investment - {name}", obj)
    for name, obj in financing_objects.items():
        _write_df(writer, f"Finance - {name}", obj)

    # Weighted-average financing terms by technology (interest rate, grace period,
    # loan term, RoE / rate of return, WACC). This is the per-technology summary table.
    _write_df(writer, "Finance - Weighted Avg by Tech", globals().get("weighted_averages"))

    # Dictionaries of per-technology outputs
    if "tech_dataframes" in globals() and isinstance(tech_dataframes, dict) and tech_dataframes:
        # tech_cashflows = pd.concat(tech_dataframes, names=["Technology", "Year"])
        _tech_clean = {
            k: v.loc[:, ~v.columns.duplicated(keep="last")]
            for k, v in tech_dataframes.items()
        }
        tech_cashflows = pd.concat(_tech_clean, names=["Technology", "Year"])
        _write_df(writer, "Technology Cashflows", tech_cashflows)
        for tech_name, df in tech_dataframes.items():
            _write_df(writer, f"Tech - {tech_name}", df)

    if "financing_requirement_by_tech" in globals() and isinstance(financing_requirement_by_tech, dict) and financing_requirement_by_tech:
        financing_req_all = pd.concat(financing_requirement_by_tech, names=["Technology", "Metric"])
        _write_df(writer, "Financing Requirement by Tech", financing_req_all)

    if "technology_disag_s1_data" in globals():
        _write_df(writer, "Technology Disag S1", technology_disag_s1_data)

    # Export index last written as first sheet-like catalog; safe even if no objects were found.
    idx_sheet = _safe_sheet_name("Export Index", used_sheet_names)
    pd.DataFrame(export_index).to_excel(writer, sheet_name=idx_sheet, index=False)

_format_workbook(output_excel_path)
print(f"Exported calculated outputs to: {output_excel_path.resolve()}")

if "all_tech_data" in globals() and isinstance(all_tech_data, dict) and all_tech_data:
    export_technology_parameter_raw_workbook(
        all_tech_data=all_tech_data,
        output_path=tech_raw_excel_path,
        years=globals().get("years"),
        scenario=scenario_label,
        variable_units=variable_units,
    )
    print(f"Exported technology raw-input table to: {tech_raw_excel_path.resolve()}")
else:
    print("Skipped technology raw-input export: `all_tech_data` was not available in the notebook state.")

_investment_blocks = {}
for _block_name, _block_df in {
    "capital_cost": globals().get("full_cap_cost_net_zero"),
    "elec_production": globals().get("df_elec_production_nz"),
    "potential_generation": globals().get("df_potential_generation_nz"),
    "opex": globals().get("df_opex"),
    # "ffe": globals().get("df_ffe"),  # Temporarily excluded: pure-input workbooks have no FFE series.
}.items():
    if isinstance(_block_df, pd.DataFrame) and not _block_df.empty:
        _investment_blocks[_block_name] = _block_df

# Weighted-average financing terms (term, grace, interest rate, RoE, WACC) by technology
_tech_fin_summary = None
try:
    from MinFin.output_export import technology_financing_summary_from_weighted_averages
    if "weighted_averages" in globals() and isinstance(weighted_averages, pd.DataFrame):
        _tech_fin_summary = technology_financing_summary_from_weighted_averages(weighted_averages)
    elif "fbs" in globals() and fbs is not None:
        _tech_fin_summary = fbs.get_technology_summary_table()
except Exception as _e:
    print(f"Skipped technology financing summary (weighted term/grace/rates/RoE/WACC): {_e}")

# Funding Availability (Excel HLD "Net Zero Funding Availability") for the pure-input path.
# Implemented components: Liabilities Payments, Cashflows, Capital Injection, Carbon Credits.
# Government Budget is a reserved interface (set `government_budget_by_year` to a per-year Series);
# it needs the legacy Funding Baseline sheet, which pure-input workbooks do not have.
from MinFin.high_level_dashboard import compute_funding_availability
from MinFin.disag_tables import _sum_tech_params
funding_availability_by_year = globals().get("funding_availability_by_year")
try:
    if funding_availability_by_year is None and "tech_dataframes" in globals() and tech_dataframes:
        _fa_cashflows = _sum_tech_params(tech_dataframes, "cashflow")
        _fa_liabilities = _sum_tech_params(tech_dataframes, "liabilities")
        _fa_carbon = None
        if "least_cost_summary" in globals() and "net_zero_summary" in globals():
            _fa_carbon = (
                (least_cost_summary["co2_emission"] - net_zero_summary["co2_emission"])
                * least_cost_summary["carbon_credit_price"]
            )
        _fa_years = [int(y) for y in (globals().get("years") or list(_fa_cashflows.index))]
        df_funding_availability = compute_funding_availability(
            _fa_years,
            cashflows=_fa_cashflows,
            liabilities_payments=_fa_liabilities,
            carbon_credits=_fa_carbon,
            capital_injection=globals().get("capital_injection"),
            government_budget=globals().get("government_budget_by_year"),  # reserved interface
        )
        funding_availability_by_year = df_funding_availability.loc["Total"]
except Exception as _e:
    print(f"Skipped funding availability computation: {_e}")

# Financing Requirement / Funding Availability as a share of GDP.
# GDP is read from the input workbook (MACROECONOMIC "Annual GDP", Mn USD). Years beyond the
# provided series are extrapolated with GDP_GROWTH_RATE. For legacy workbooks (no MACROECONOMIC),
# set ANNUAL_GDP_MN_USD (base-year GDP) to enable the rows via growth-rate extrapolation.
from MinFin.excel_io import read_annual_gdp
GDP_GROWTH_RATE = globals().get("GDP_GROWTH_RATE", 0.0526)
ANNUAL_GDP_MN_USD = globals().get("ANNUAL_GDP_MN_USD")
_economy_metrics_for_export = {}
try:
    _gdp_years = [int(y) for y in (globals().get("years") or [])]
    _gdp_input = read_annual_gdp(globals().get("file_path", ""))
    _gdp_by_year = pd.Series(dtype=float)
    if _gdp_years and not _gdp_input.empty:
        _base_year = int(_gdp_input.index.min())
        _base_val = float(_gdp_input.loc[_base_year])
        _gdp_by_year = pd.Series({
            y: (float(_gdp_input.loc[y]) if y in _gdp_input.index
                else _base_val * (1 + GDP_GROWTH_RATE) ** (y - _base_year))
            for y in _gdp_years
        })
    elif _gdp_years and ANNUAL_GDP_MN_USD:
        _b = _gdp_years[0]
        _gdp_by_year = pd.Series(
            {y: ANNUAL_GDP_MN_USD * (1 + GDP_GROWTH_RATE) ** (y - _b) for y in _gdp_years}
        )
    elif _gdp_years:
        print("Skipped share-of-GDP rows: no 'Annual GDP' in MACROECONOMIC and ANNUAL_GDP_MN_USD not set.")

    if not _gdp_by_year.empty:
        def _year_series(obj):
            s = pd.Series(obj)
            return pd.Series(
                {int(c): pd.to_numeric(v, errors="coerce")
                 for c, v in s.items() if str(c).strip().isdigit()}
            )

        if "financing_summary" in globals() and "Financing Requirement" in getattr(financing_summary, "index", []):
            _fr = _year_series(financing_summary.loc["Financing Requirement"])
            _idx = _gdp_by_year.index.intersection(_fr.index)
            if len(_idx):
                _economy_metrics_for_export["financing_requirement_share_of_gdp"] = (
                    _fr.loc[_idx] / _gdp_by_year.loc[_idx]
                ).sort_index()

        # Funding availability Total computed above (Government Budget left as a reserved
        # interface); divide by GDP for the share-of-GDP row.
        _fa = funding_availability_by_year
        if isinstance(_fa, pd.Series) and not _fa.empty:
            _fa = _year_series(_fa)
            _idx = _gdp_by_year.index.intersection(_fa.index)
            if len(_idx):
                _economy_metrics_for_export["funding_availability_share_of_gdp"] = (
                    _fa.loc[_idx] / _gdp_by_year.loc[_idx]
                ).sort_index()
except Exception as _e:
    print(f"Skipped share-of-GDP rows: {_e}")

if "tech_dataframes" in globals() and isinstance(tech_dataframes, dict) and tech_dataframes:
    tech_dataframes = {
        k: v.loc[:, ~v.columns.duplicated(keep="last")]
        for k, v in tech_dataframes.items()
    }
    assert hasattr(_minfin_output_export, "_convert_allocation_values_to_display"), "Reload MinFin.output_export failed — restart kernel"
    print(
        "Export FX:",
        "display=", getattr(fb, "display_currency", None),
        "local=", fb.currency,
        "foreign=", fb.foreign_currency,
        "rates_cols=", list(getattr(exchange_rates, "columns", [])),
    )
    export_technology_output_workbook(
        tech_dataframes=tech_dataframes,
        output_path=tech_output_excel_path,
        all_tech_data=globals().get("all_tech_data"),
        financing_requirement_by_tech=globals().get("financing_requirement_by_tech"),
        investment_blocks=_investment_blocks or None,
        df_technologies=globals().get("df_technologies"),
        variable_units=variable_units,
        years=globals().get("years"),
        repayment_schedule=globals().get("repayment_schedule"),
        technology_financing_summary=_tech_fin_summary,
        existing_financing_by_technology=_existing_fin_by_tech,
        economy_metrics=_economy_metrics_for_export or None,
        scenario=scenario_label,
        display_currency=getattr(fb, "display_currency", None) or "USD",
        local_currency_code=fb.currency,
        foreign_currency_code=fb.foreign_currency,
        exchange_rates=exchange_rates,
    )
    print(f"Exported full technology output table to: {tech_output_excel_path.resolve()}")
else:
    print("Skipped full technology output export: `tech_dataframes` was not available in the notebook state.")


Exported calculated outputs to: /Users/zl17868/Library/CloudStorage/Dropbox/CCG/MinFin/minfin_output/minfin_calculated_outputs.xlsx
Exported technology raw-input table to: /Users/zl17868/Library/CloudStorage/Dropbox/CCG/MinFin/minfin_output/minfin_technology_raw_input.xlsx
Export FX: display= USD local= KES foreign= USD rates_cols= ['USD', 'KES']
Exported full technology output table to: /Users/zl17868/Library/CloudStorage/Dropbox/CCG/MinFin/minfin_output/minfin_technology_output_in_osemosys_way.xlsx


In [190]:
# Standalone export: technology tables (inputs + full computed outputs)
from pathlib import Path
import pandas as pd
from MinFin.excel_io import load_workbook_variable_units
import importlib
import MinFin.output_export as _minfin_output_export
importlib.reload(_minfin_output_export)
from MinFin.output_export import (
    export_technology_output_workbook,
    export_technology_parameter_raw_workbook,
)
variable_units = load_workbook_variable_units(file_path)

output_dir = Path("minfin_output")
tech_raw_excel_path = output_dir / "minfin_technology_raw_input.xlsx"
tech_output_excel_path = output_dir / "minfin_technology_output_in_osemosys_way.xlsx"

if "all_tech_data" in globals() and isinstance(all_tech_data, dict) and all_tech_data:
    export_technology_parameter_raw_workbook(
        all_tech_data=all_tech_data,
        output_path=tech_raw_excel_path,
        years=globals().get("years"),
        scenario=scenario_label,
        variable_units=variable_units,
    )
    print(f"Exported technology raw-input table to: {tech_raw_excel_path.resolve()}")
else:
    print("Skipped export: `all_tech_data` is not available. Run the technology extraction cells first.")

_investment_blocks = {}
for _block_name, _block_df in {
    "capital_cost": globals().get("full_cap_cost_net_zero"),
    "elec_production": globals().get("df_elec_production_nz"),
    "potential_generation": globals().get("df_potential_generation_nz"),
    "opex": globals().get("df_opex"),
    # "ffe": globals().get("df_ffe"),  # Temporarily excluded: pure-input workbooks have no FFE series.
}.items():
    if isinstance(_block_df, pd.DataFrame) and not _block_df.empty:
        _investment_blocks[_block_name] = _block_df

# Weighted-average financing terms (term, grace, interest rate, RoE, WACC) by technology
_tech_fin_summary = None
try:
    from MinFin.output_export import technology_financing_summary_from_weighted_averages
    if "weighted_averages" in globals() and isinstance(weighted_averages, pd.DataFrame):
        _tech_fin_summary = technology_financing_summary_from_weighted_averages(weighted_averages)
    elif "fbs" in globals() and fbs is not None:
        _tech_fin_summary = fbs.get_technology_summary_table()
except Exception as _e:
    print(f"Skipped technology financing summary (weighted term/grace/rates/RoE/WACC): {_e}")

# Funding Availability (Excel HLD "Net Zero Funding Availability") for the pure-input path.
# Implemented components: Liabilities Payments, Cashflows, Capital Injection, Carbon Credits.
# Government Budget is a reserved interface (set `government_budget_by_year` to a per-year Series);
# it needs the legacy Funding Baseline sheet, which pure-input workbooks do not have.
from MinFin.high_level_dashboard import compute_funding_availability
from MinFin.disag_tables import _sum_tech_params
funding_availability_by_year = globals().get("funding_availability_by_year")
try:
    if funding_availability_by_year is None and "tech_dataframes" in globals() and tech_dataframes:
        _fa_cashflows = _sum_tech_params(tech_dataframes, "cashflow")
        _fa_liabilities = _sum_tech_params(tech_dataframes, "liabilities")
        _fa_carbon = None
        if "least_cost_summary" in globals() and "net_zero_summary" in globals():
            _fa_carbon = (
                (least_cost_summary["co2_emission"] - net_zero_summary["co2_emission"])
                * least_cost_summary["carbon_credit_price"]
            )
        _fa_years = [int(y) for y in (globals().get("years") or list(_fa_cashflows.index))]
        df_funding_availability = compute_funding_availability(
            _fa_years,
            cashflows=_fa_cashflows,
            liabilities_payments=_fa_liabilities,
            carbon_credits=_fa_carbon,
            capital_injection=globals().get("capital_injection"),
            government_budget=globals().get("government_budget_by_year"),  # reserved interface
        )
        funding_availability_by_year = df_funding_availability.loc["Total"]
except Exception as _e:
    print(f"Skipped funding availability computation: {_e}")

# Financing Requirement / Funding Availability as a share of GDP.
# GDP is read from the input workbook (MACROECONOMIC "Annual GDP", Mn USD). Years beyond the
# provided series are extrapolated with GDP_GROWTH_RATE. For legacy workbooks (no MACROECONOMIC),
# set ANNUAL_GDP_MN_USD (base-year GDP) to enable the rows via growth-rate extrapolation.
from MinFin.excel_io import read_annual_gdp
GDP_GROWTH_RATE = globals().get("GDP_GROWTH_RATE", 0.0526)
ANNUAL_GDP_MN_USD = globals().get("ANNUAL_GDP_MN_USD")
_economy_metrics_for_export = {}
try:
    _gdp_years = [int(y) for y in (globals().get("years") or [])]
    _gdp_input = read_annual_gdp(globals().get("file_path", ""))
    _gdp_by_year = pd.Series(dtype=float)
    if _gdp_years and not _gdp_input.empty:
        _base_year = int(_gdp_input.index.min())
        _base_val = float(_gdp_input.loc[_base_year])
        _gdp_by_year = pd.Series({
            y: (float(_gdp_input.loc[y]) if y in _gdp_input.index
                else _base_val * (1 + GDP_GROWTH_RATE) ** (y - _base_year))
            for y in _gdp_years
        })
    elif _gdp_years and ANNUAL_GDP_MN_USD:
        _b = _gdp_years[0]
        _gdp_by_year = pd.Series(
            {y: ANNUAL_GDP_MN_USD * (1 + GDP_GROWTH_RATE) ** (y - _b) for y in _gdp_years}
        )
    elif _gdp_years:
        print("Skipped share-of-GDP rows: no 'Annual GDP' in MACROECONOMIC and ANNUAL_GDP_MN_USD not set.")

    if not _gdp_by_year.empty:
        def _year_series(obj):
            s = pd.Series(obj)
            return pd.Series(
                {int(c): pd.to_numeric(v, errors="coerce")
                 for c, v in s.items() if str(c).strip().isdigit()}
            )

        if "financing_summary" in globals() and "Financing Requirement" in getattr(financing_summary, "index", []):
            _fr = _year_series(financing_summary.loc["Financing Requirement"])
            _idx = _gdp_by_year.index.intersection(_fr.index)
            if len(_idx):
                _economy_metrics_for_export["financing_requirement_share_of_gdp"] = (
                    _fr.loc[_idx] / _gdp_by_year.loc[_idx]
                ).sort_index()

        # Funding availability Total computed above (Government Budget left as a reserved
        # interface); divide by GDP for the share-of-GDP row.
        _fa = funding_availability_by_year
        if isinstance(_fa, pd.Series) and not _fa.empty:
            _fa = _year_series(_fa)
            _idx = _gdp_by_year.index.intersection(_fa.index)
            if len(_idx):
                _economy_metrics_for_export["funding_availability_share_of_gdp"] = (
                    _fa.loc[_idx] / _gdp_by_year.loc[_idx]
                ).sort_index()
except Exception as _e:
    print(f"Skipped share-of-GDP rows: {_e}")

if "tech_dataframes" in globals() and isinstance(tech_dataframes, dict) and tech_dataframes:
    assert hasattr(_minfin_output_export, "_convert_allocation_values_to_display"), "Reload MinFin.output_export failed — restart kernel"
    print(
        "Export FX:",
        "display=", getattr(fb, "display_currency", None),
        "local=", fb.currency,
        "foreign=", fb.foreign_currency,
        "rates_cols=", list(getattr(exchange_rates, "columns", [])),
    )
    export_technology_output_workbook(
        tech_dataframes=tech_dataframes,
        output_path=tech_output_excel_path,
        all_tech_data=globals().get("all_tech_data"),
        financing_requirement_by_tech=globals().get("financing_requirement_by_tech"),
        investment_blocks=_investment_blocks or None,
        df_technologies=globals().get("df_technologies"),
        variable_units=variable_units,
        years=globals().get("years"),
        repayment_schedule=globals().get("repayment_schedule"),
        technology_financing_summary=_tech_fin_summary,
        economy_metrics=_economy_metrics_for_export or None,
        scenario=scenario_label,
        display_currency=getattr(fb, "display_currency", None) or "USD",
        local_currency_code=fb.currency,
        foreign_currency_code=fb.foreign_currency,
        exchange_rates=exchange_rates,
    )
    print(f"Exported full technology output table to: {tech_output_excel_path.resolve()}")
else:
    print("Skipped full technology output export: run cashflow / financing cells first.")


Exported technology raw-input table to: /Users/zl17868/Library/CloudStorage/Dropbox/CCG/MinFin/minfin_output/minfin_technology_raw_input.xlsx
Export FX: display= USD local= KES foreign= USD rates_cols= ['USD', 'KES']
Exported full technology output table to: /Users/zl17868/Library/CloudStorage/Dropbox/CCG/MinFin/minfin_output/minfin_technology_output_in_osemosys_way.xlsx


# High Level Dashboard

From this cell we do the statistics for high level dashboard.
- First we import necesssary modules from MinFin  


In [191]:
from MinFin.utils import *
from MinFin import high_level_dashboard,EconomicParameters,Scenarios,CapitalInjection

 - Then we set up the necessary parameters, by assigning values to several dataclass defined in the MinFin Module 


In [192]:
econ_params = EconomicParameters(
    income_elasticity_of_energy_demand=0.7, 
    cagr_of_real_energy_price=0.01, 
    gdp_growth_rate=0.0526
)

capital_injection =  CapitalInjection(
    money_from = "domestic public",
    start_year = 2024,
    volume = 100.0,
    duration = 3,
    type = "government budget"
    )

scenarios = Scenarios(
    financial_instrument_type = "Grant",
    scenario = "NetZero",
    capital_injection = capital_injection,
)
        

In [193]:
# See next cell: DisagLookupConfig + build_disag_table from MinFin.disag_tables


 - Updates: we add technology specific waccs here: 


In [194]:
from MinFin.disag_tables import DisagLookupConfig, build_disag_table

cfg = DisagLookupConfig(financing_source="Comm_Intl")
df = build_disag_table(cfg, technology_disag_s1_data)
df


Category              Debt                                Equity               \
Parameter    Interest Rate Grace Period Loan Term Rate of Return Project Life   
Battery           0.186644            0     15.77       0.224844           40   
Biomass           0.180044            0     15.77       0.218244           30   
Geothermal        0.175744            0     15.77       0.213944           25   
Hydropower        0.172744            0     15.77       0.200244           50   
Solar PV          0.176144            0     15.77       0.214344           24   
Wind              0.172744            0     15.77       0.202944           25   
Imports           0.000000            0      0.00       0.000000            0   
Nuclear           0.176244            0     15.77       0.214444           60   
Gas               0.174144            0     15.77       0.212344           30   
Oil               0.000000            0      0.00       0.000000            0   
Transmission      0.176244            0     15.77       0.214444           50   
Distribution      0.176244            0     15.77       0.214444           70   

Category     Financing Shares                                \
Parameter          Debt Share Equity Share Share of Finance   
Battery                0.6302       0.3698         0.044147   
Biomass                0.6302       0.3698         0.006079   
Geothermal             0.6302       0.3698         0.530562   
Hydropower             0.6302       0.3698         0.931010   
Solar PV               0.6302       0.3698         1.000000   
Wind                   0.6302       0.3698         0.476632   
Imports                0.0000       0.0000         0.000000   
Nuclear                0.6302       0.3698         1.000000   
Gas                    0.6302       0.3698         0.389871   
Oil                    0.0000       0.0000         0.000000   
Transmission           0.6302       0.3698         0.492531   
Distribution           0.6302       0.3698         0.559947   

Category     Foreign Currency Shares         
Parameter                       Debt Equity  
Battery                            1      1  
Biomass                            1      1  
Geothermal                         1      1  
Hydropower                         1      1  
Solar PV                           1      1  
Wind                               1      1  
Imports                            0      0  
Nuclear                            1      1  
Gas                                1      1  
Oil                                0      0  
Transmission                       1      1  
Distribution                       1      1

In [195]:
# hd = high_level_dashboard(repayment_statistics,econ_params,scenarios,df_funding_baseline_full=df_funding_baseline_full,least_cost_summary=least_cost_summary,net_zero_summary=net_zero_summary)

# ## Example usage of the class

# hd.get_funding_availability_lever()
# hd.get_co2_savings(least_cost_summary,net_zero_summary)
# hd.get_financing_source_shares()

 - Two rows in the sheet are user inputs, they need to be read in. 


In [196]:
# import numpy as np
# from MinFin.disag_tables import get_row_by_name

# df_hd = pd.read_excel(
#     file_path,
#     sheet_name="High Level Dashboard",
#     index_col=0,
#     engine="openpyxl",
# )
# liability_payments = get_row_by_name(df_hd, "Liabilities Payments", start_year=START_YEAR)
# annual_gdp = get_row_by_name(df_hd, "Annual GDP", start_year=START_YEAR)

# liability_payments


In [197]:
# nz_needs = hd.get_net_zero_financing_needs_full(df_invest_need_summary,df_funding_envelope) # Net Zero financing needs
# lc_needs = hd.get_least_cost_financing_needs_full(df_invest_need_summary,df_funding_envelope) # Least Cost financing needs
# additional_investment_needs = hd.get_additional_investment_needs(lc_needs,nz_needs) #Additional needs (NetZero - LeastCost)
# # nz_needs.head(6)
# needs_of_different_scenario = {"NetZero":nz_needs,"LeastCost":lc_needs,"Incremental":additional_investment_needs} #put them in one dict for further processing 


# dashboard_summary= hd.get_summary(df_funding_availability_full,df_invest_need_summary,needs_of_different_scenario,repayment_schedule)

# financing_sources = hd.get_funding_sources(dashboard_summary)

# df_financing_repay

# ments= hd.get_repayments(repayment_schedule,df_invest_need_summary,df_funding_envelope)

# df_debt_stock = hd.get_debt_stock(dashboard_summary,nz_needs)

# gdp_percentage = hd.get_gdp_percentage(dashboard_summary)


In [198]:
# from MinFin.high_level_dashboard import hd

# hd = hd(
#     repayment_statistics,
#     econ_params,
#     scenarios,
#     start_year=2025,
#     least_cost_summary=least_cost_summary,
#     net_zero_summary=net_zero_summary,
#     financing_summary=financing_summary,
# )
# df_funding_availability_full = hd.get_funding_availability_full(df_funding_envelope, financing_summary)
# df_funding_availability_full.loc["Liabilities Payments"] = liability_payments.T

# df_funding_availability_full.loc["Total", :] -= df_funding_availability_full.loc["Liabilities Payments"]

# df_funding_availability_full


In [199]:
# existing_repayments= hd.get_existing_financing(repayment_schedule)
# existing_repayments_named = existing_repayments.rename("existing_financing_requirement")
# liability_payments_named = liability_payments.rename("liabilities")
# financing_summary_filtered = financing_summary[
#     ~financing_summary.index.str.contains("Cashflows|Generation|Exports|Transmission|Distribution")
# ]
# hd_financing_requirements = pd.concat(
#     [existing_repayments_named, financing_summary_filtered.T, liability_payments_named.T], axis=1
# )
# hd_financing_requirements["total_financing"] = hd_financing_requirements[["Financing Requirement", "liabilities","existing_financing_requirement"]].sum(axis=1)
# financing_summary

In [200]:
# df_invest_need_summary 
# # df_funding_envelope
# # total_financing_needs_nz
# dashboard_summary=pd.DataFrame()
# dashboard_summary["investment_need"]=df_invest_need_summary["Net Zero"]
# dashboard_summary["funding_availability"]=df_funding_availability_full.loc["Total"]
# dashboard_summary["financing_requirement"] = hd_financing_requirements["total_financing"]
# dashboard_summary["funding_shortfall"] = dashboard_summary["financing_requirement"] - dashboard_summary["funding_availability"]
# dashboard_summary=dashboard_summary.T

In [201]:

# df_macro = pd.DataFrame({
#     "annual_gdp": annual_gdp,
#     # "Annual GDP Foreign Currency (Million USD)": annual_gdp,
#     "financing_requirement/gdp": dashboard_summary.loc["financing_requirement",:] / annual_gdp,
#     "funding_availability/gdp": dashboard_summary.loc["funding_availability",:] / annual_gdp,
#     # "Funding Availability (% GDP)": fund_musd / annual_gdp,
# })
# df_macro.T


 - Tech Look up
-  


In [202]:
# import matplotlib.pyplot as plt
# import matplotlib.ticker as mticker
# from MinFin.plotting_notebook import fmt_million_usd

# tech = "Biomass"
# years = tech_dataframes[tech]["investment_need"].index

# fig, ax = plt.subplots(figsize=(12, 6))
# fig.patch.set_facecolor("white")
# ax.set_facecolor("white")
# inv = tech_dataframes[tech]["investment_need"]
# cf = tech_dataframes[tech]["cashflow"]
# fr = financing_requirement_by_tech[tech].T["Financing Requirement"]
# ax.plot(years, inv, color="#2ca02c", linestyle="-", linewidth=2.0, zorder=3, label="Investment Need")
# ax.plot(years, cf, color="#1f77b4", linestyle="--", linewidth=2.4, zorder=5, label="Cashflow")
# ax.plot(years, fr, color="#ff7f0e", linestyle="-.", linewidth=2.4, zorder=4, label="Financing Requirement")
# ax.axhline(0, color="grey", linewidth=2.5, zorder=2)
# ax.set_xlabel("Year")
# ax.set_ylabel("Million USD")
# ax.xaxis.set_major_locator(mticker.MultipleLocator(5))
# ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_million_usd))
# ax.grid(axis="y", color="lightgray", linewidth=0.8)
# ax.set_axisbelow(True)
# for spine in ax.spines.values():
#     spine.set_visible(False)
# ax.legend(loc="best", frameon=False)
# plt.tight_layout()
# plt.show()


In [203]:
# import seaborn as sns
# import matplotlib.pyplot as plt
# import pandas as pd
# from MinFin.plotting_notebook import plot_metric_time_series

# plot_metric_time_series(
#     dashboard_summary,
#     selected_rows=["financing_requirement", "investment_need", "funding_availability"],
#     title="Financing Requirement and Funding Shortfall over Years",
# )
# plot_metric_time_series(
#     dashboard_summary / annual_gdp,
#     selected_rows=["financing_requirement", "funding_availability"],
#     title="Financing Requirement and Funding Requirement relative to GDP",
# )


In [204]:
# import seaborn as sns
# import matplotlib.pyplot as plt
# import pandas as pd

# # Assume df_result structure is similar to:
# #["NetZero Investment Needs (Million USD)", "NetZero Financing Requirements (Million USD)", ...]
# # Columns: Years such as 2025, 2026, ...
# # Each cell represents the value of the corresponding metric for that year

# # 1. Select the 5 rows we need to plot, and transpose to make "Year" the row index
# df_plot = pd.DataFrame({
#                     "LC Investment needs": lc_needs.loc["Investment needs",:],
#                     "LC Financing requirement": lc_needs.loc["Financing Requirements",:],
#                     "NZ Investment needs": nz_needs.loc["Investment needs",:],
#                     "NZ Financing requirement": nz_needs.loc["Financing Requirements",:],
#                     "Funding availability": dashboard_summary.loc["funding_availability",:],
#                 })

# # 2. Reset index and convert to long format for easier plotting with seaborn
# df_plot = df_plot.reset_index().rename(columns={"index": "Year"})
# df_long = df_plot.melt(id_vars="Year", var_name="Metric", value_name="Value")

# # If Year is str, cast to int if needed
# # df_long["Year"] = df_long["Year"].astype(int)

# # 3. Seaborn style and plot
# sns.set(style="whitegrid")
# plt.figure(figsize=(12, 6))

# # Line colors (adjust as needed)
# palette = {
#     "LC Investment needs": "black",
#     "LC Financing requirement": "red",
#     "Funding availability": "green",
#     "NZ Investment needs": "gray",
#     "NZ Financing requirement": "pink",
# }

# ax = sns.lineplot(
#     data=df_long,
#     x="Year",
#     y="Value",
#     hue="Metric",
#     palette=palette,
#     marker="o",
#     linewidth=2
# )

# # 4. Set title, axis labels, and legend
# ax.set_title("NetZero & LeastCost Investment/Financing Over Years", fontsize=16)
# ax.set_xlabel("Year", fontsize=14)
# ax.set_ylabel("Amount (Million USD)", fontsize=14)
# plt.legend(title="Metric", fontsize=12, title_fontsize=14)

# plt.tight_layout()
# # plt.show()



In [205]:
import pandas as pd
import plotly.express as px
from MinFin.plotting_notebook import plot_co2_emissions_scenarios

_idx = net_zero_summary.index
if _IS_PURE_INPUT_WB:
    from MinFin.data_processor import emission_savings_series_from_investment_plan

    _emissions_savings = emission_savings_series_from_investment_plan(file_path, _idx)
else:
    _emissions_savings = -net_zero_summary["co2_emission"] + least_cost_summary["co2_emission"]

df_emissions = pd.DataFrame(
    {
        "Least Cost": least_cost_summary["co2_emission"].reindex(_idx, fill_value=0),
        "Net Zero": net_zero_summary["co2_emission"],
        "Emissions Savings": _emissions_savings,
    },
    index=_idx,
)

plot_co2_emissions_scenarios(df_emissions)


In [206]:
# df_funding_availability_full

In [207]:
# import plotly.graph_objects as go
# import plotly.express as px
# import pandas as pd
# from MinFin.plotting_notebook import stacked_area_fig, DEFAULT_STACK_COLOR_MAP as color_map

# width = 800
# height = 600
# years = list(dashboard_summary.columns)

# df_funding_plot = pd.DataFrame(
#     {
#         "Year": dashboard_summary.columns,
#         "Government Budget": list(df_funding_availability_full.loc["Government Budget", years]),
#         "Cashflows": list(df_funding_availability_full.loc["Cashflows", years]),
#         "Capital injection": list(df_funding_availability_full.loc["Capital Injection", years]),
#         "Carbon Credits": list(df_funding_availability_full.loc["Carbon Credits", years]),
#         "Financing Requirement": dashboard_summary.loc["financing_requirement", years],
#         "Funding Availability": dashboard_summary.loc["funding_availability", years],
#     }
# )

# stacked_cols_funding = [
#     "Government Budget",
#     "Cashflows",
#     "Capital injection",
#     "Carbon Credits",
# ]
# title = "Funding Availability"
# fig = stacked_area_fig(
#     df_funding_plot,
#     stacked_cols_funding,
#     width=800,
#     height=600,
#     color_map=color_map,
#     title=title,
#     line_plots=True,
# )
# fig.show()


In [208]:
# import pandas as pd
# import plotly.offline as pyo
# from MinFin.plotting_notebook import plot_funding_shortfall_bar_interactive

# pyo.init_notebook_mode(connected=True)
# plot_funding_shortfall_bar_interactive(dashboard_summary, row_label="funding_shortfall")


In [209]:
# incremental_investment
net_zero_summary.head()
# df_5yr_avg["delta_capital_cost"].values
# df_financing_needs_ffr
# df_category_sum_nz
incremental_investment=df_category_sum_incremental
incremental_investment.columns


Index(['Distribution Infrastructure', 'Generation Fossil-Fuel',
       'Generation Renewable', 'Transmission Infrastructure', 'Battery',
       'Biomass', 'Distribution', 'Gas', 'Geothermal', 'Hydropower', 'Imports',
       'Nuclear', 'Oil', 'Solar PV', 'Transmission', 'Wind'],
      dtype='object', name='Technology')

In [210]:
# import pandas as pd
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots
# import plotly.express as px

# width = 800
# height = 600
# df_long_nz = df_category_sum_nz.reset_index().melt(
#     id_vars="Year", 
#     var_name="Technology", 
#     value_name="InvestmentNeed"
# )
# df_long_incremental = incremental_investment.reset_index().melt(
#     id_vars="Year", 
#     var_name="Technology", 
#     value_name="InvestmentNeed"
# )

# # ==========  2 rows × 4 columns  ==========
# fig = make_subplots(
#     rows=2, cols=4,
#     subplot_titles=[
#         "Plot 1: Investment Needs",       # Subplot 1 title
#         "Plot 2: Cost of Capital",        # Subplot 2 title
#         "Plot 3: Borrowed Sources",       # Subplot 3 title
#         "Plot 4: Scenario Difference",    # Subplot 4 title
#         "Plot 5: Future Needs by Source", # Subplot 5 title
#         "Plot 6: Funding Shortfall",      # Subplot 6 title
#         "Plot 7: Additional Metrics",     # Subplot 7 title
#         "Plot 8: Another Metric"          # Subplot 8 title
#     ]
# )

# color_map = {

#     "Biomass": "#ff9900",       # Orange
#     "Geothermal": "#66ccff",    # Light Blue
#     "Solar PV": "#ffd700",      # Gold
#     "Wind": "#66ff66",          # Light Green

#     "Nuclear": "#999999",       # Gray
    
#     "Hydropower": "#3399FF",    # Blue
#     "CSP": "#FFB84D",           # Light Orange
#     "Oil": "#808080",  # Medium Gray
#     "Direct Oil": "#9966FF",  # Purple
#     "Other": "#BEBEBE"          # Silver Gray
# }

# df_long_nz['Year'] = df_long_nz['Year'].astype(int).astype(str)

# fig_area1 = px.area(
#     df_long_nz, 
#     x="Year", 
#     y="InvestmentNeed", 
#     color="Technology", 
#     color_discrete_map=color_map,
#     title="Projected Investment Needs to Reach Net Zero by 2050",
#     labels={"InvestmentNeed": "Million US$", "Year": "Year"}
# )
# fig_area1.update_layout(
#     template="plotly_white",
#     showlegend=True,
#     width=width,
#     height=height
#     )

# fig_area2  = px.bar(
#     df_long_incremental, 
#     x="Year",
#     y="InvestmentNeed",
#     color="Technology",
#     orientation="v",
#     color_discrete_map=color_map,
#     barmode="relative", 
#     title="Additional Capital Cost of Net Zero",
#     labels={"InvestmentNeed": "Million US$", "Year": "Year"}
# )

# fig_area2.update_layout(
#     template="plotly_white",
#     xaxis=dict(range=[2024.5, 2070.5]),
#     yaxis=dict(zeroline=True, zerolinewidth=2, zerolinecolor="black"),
#     width=width,
#     height=height
# )
# fig_area2.update_layout(
#     template="plotly_white",    
#     )


# df_cost = pd.DataFrame({
#     "Scenario": ["NetZero", "NetZero", "NetZero"],
#     "Cost Type": ["Capital Cost", "O&M Costs (Variable)", "O&M Costs (Fixed)"],
#     "Value": [
#         net_zero_summary["capital_cost"].sum(),
#         net_zero_summary["variable_cost"].sum(),
#         net_zero_summary["fixed_cost"].sum()
#         ]  # Million US$
#     })

# fig_area3 = px.bar(
#     df_cost,
#     x="Scenario",
#     y="Value",
#     color="Cost Type",
#     barmode="stack",  
#     title="Total Undiscounted Scenario Cost 2025 to 2070",
#     labels={"Value": "Million US$", "Scenario": ""}
# )

# fig_area3.update_layout(
#     template="plotly_white",
#     bargap=0, 
#     showlegend=True,
#     width=width,
#     height=height
#     )

# df_diff = pd.DataFrame({
#     "Year": df_5yr_avg["Year"],
#     "Capital Cost":  list(df_5yr_avg["delta_capital_cost"].values),
#     "Variable Cost": list(df_5yr_avg["delta_variable_cost"].values),
#     "Fixed Cost":    list(df_5yr_avg["delta_fixed_cost"].values),
#     "Cost of Carbon":list(df_5yr_avg["emission_saving"].values),
#     # "Net Benefits":  df_5yr_avg["delta_capital_cost"]
# })
# # df_diff = pd.DataFrame({
# #     "Year": df_5yr_avg["Year"],
# #     "Capital Cost": list(df_5yr_avg["delta_capital_cost"].values),
# #     "Variable Cost": [0,   20, 8,   0,   30,  0,   0,   0,   0  ],
# #     "Fixed Cost":    [0,   0,  7,   0,   0,   0,   0,   0,   0  ],
# #     "Cost of Carbon":[0,   0,  0,   30,  37,  40,  50,  60,  70 ],
# #     "Net Benefits":  [-46, 20, 15, 47,  67, 107, 221, 275, 370]
# # })
# df_diff_long= df_diff.melt(
#     id_vars=["Year"],  
#     value_vars=["Capital Cost", "Variable Cost", "Fixed Cost", "Cost of Carbon"],
#     var_name="Cost Type",
#     value_name="Value"
# )
# df_diff["Net Benefits"] = df_diff["Capital Cost"] + df_diff["Variable Cost"] + df_diff["Fixed Cost"] + df_diff["Cost of Carbon"]

# color_map_costs = {

#     "Cost of Carbon": "#66ff66",          # Light Green

#     "Variable Cost": "#cc6600",           # Dark Orange

#     "Capital Cost": "#3399FF",    # Blue
#     "Fixed Cost":    "#66ccff"   # Light Blue
# }
# # stacked bar chart
# fig_area4 = px.bar(
#     df_diff_long,
#     x="Year",
#     y="Value",
#     color="Cost Type",
#     color_discrete_map=color_map_costs,
#     barmode="relative",  
#     title="Differential benefits and costs of NZ vs BAU (Million US$)",
#     labels={"Value": "Million US$", "Year": "Year"}
# )
# fig_area4.add_trace(
#     go.Scatter(
#         x=df_diff["Year"],           
#         y=df_diff["Net Benefits"],   # Net Benefits
#         mode="markers+text",    
#         text=df_diff["Net Benefits"].apply(lambda x: f"{round(x)}").astype(str),  
#         textposition="top center",            
#         marker=dict(
#             symbol=141,       # horizontal line symbol 'line-ew', 141, '141'
#             size=20,                # the length of the line is determined by size
#             color="red",
#         ),
#         name="Net Benefits"     
#     )
# )
# fig_area4.update_layout(
#     template="plotly_white",
#     showlegend=True,
#     xaxis=dict(range=[2020.5, 2070.5]),
#     width=width,
#     height=height
#     )


# # 1) data 
# df_stranded_cost = pd.DataFrame({
#     "Fuel": ["Coal", "Gas", "Oil"],
#     "Extra Stranded Cost": [0, 0, 0]  # initialised with 0, will be updated later
# })
# since_year = 2025
# for tech in  df_stranded_cost["Fuel"]:
#     # Calculate cumulative stranded costs from since_year
#     df_stranded_cost.loc[df_stranded_cost["Fuel"]==tech, "Extra Stranded Cost"]=df_financing_needs_ffr.loc[df_financing_needs_ffr.index >= since_year,(tech,tech)].sum()
# # 2) Draw a bar chart
# fig_stranded_cost = px.bar(
#     df_stranded_cost,
#     x="Fuel",
#     y="Extra Stranded Cost",
#     title="Extra Stranded Cost Compensation of Net Zero",
#     labels={"Extra Stranded Cost": "Million US$", "Fuel": ""},
#     text="Extra Stranded Cost"  # show values on the bar
# )

    
# df_plot = pd.DataFrame({
#     "Year": df_category_sum_nz.reset_index()["Year"],
#     "Net Zero Financing Needs": df_category_sum_nz.copy().reset_index().sum(axis=1),
# })
# # Convert both columns to the same data type (float)
# df_plot["Net Zero Financing Needs"] = df_plot["Net Zero Financing Needs"].astype(float)

# fig_needs_by_source = px.bar(
#     df_plot,
#     x="Year",
#     y=["Net Zero Financing Needs"],
#     barmode="relative",
#     color_discrete_sequence=["grey", "black"],
#     title="Future Investment Needs by Source",
#     labels={"value": "Million US$", "variable": ""},  # adjust the legend/axis labels
#     template="plotly_white",
# )

# # add black border to all bars
# fig_needs_by_source.update_traces(marker=dict(line=dict(color="black", width=1)))

# # adjust layout, e.g. move the legend below, change the width and height
# fig_needs_by_source.update_layout(
#     legend=dict(
#         orientation="h",
#         yanchor="bottom",
#         y=-0.15,  # legend below
#         xanchor="center",
#         x=0.5
#     ),
#     xaxis=dict(range=[2024, 2070.5]),
    
#     width=width,
#     height=height
# )

# df = pd.DataFrame({
#     "Year": net_zero_summary.index,
#     "NetZero": net_zero_summary["cost_of_elec"],
#     "LeastCost": least_cost_summary["cost_of_elec"],
# })

# # melt NetZero and LeastCost into "Scenario" and "Cost" two columns
# df_long_elec_cost = df.melt(
#     id_vars="Year",
#     var_name="Scenario",
#     value_name="Cost"
# )
# # now the structure of df_long is similar to:
# #   Year   Scenario   Cost
# # 0 2025   NetZero    14
# # 1 2030   NetZero    15
# # 2 2035   NetZero    12
# # ... 
# # . 2025   LeastCost  14

# fig_elec_cost = px.line(
#     df_long_elec_cost,
#     x="Year",
#     y="Cost",
#     color="Scenario",
#     title="Average Cost of Electricity US$/TWh",
#     labels={"Cost": "US$/TWh", "Year": "Year", "Scenario": ""},
#     color_discrete_map={"NetZero": "green", "LeastCost": "aqua"}  # customise colors
# )

# fig_elec_cost.update_layout(
#     template="plotly_white",
#     # yaxis=dict(range=[0, 20]),  # according to your data range, or let Plotly decide
#     width=width,
#     height=height
# )


# df_capital = pd.DataFrame({
#     "Year": net_zero_summary.index,
#     "NetZero": net_zero_summary["capital_cost"],
#     "LeastCost": least_cost_summary["capital_cost"],
# })

# # melt NetZero and LeastCost into "Scenario" and "Cost" two columns
# df_capital_cost = df_capital.melt(
#     id_vars="Year",
#     var_name="Scenario",
#     value_name="Cost"
# )

# fig_capital_cost = px.line(
#     df_capital_cost,
#     x="Year",
#     y="Cost",
#     color="Scenario",
#     title="Capital Cost NZ vs LC",
#     labels={"Cost": "Million US$", "Year": "Year", "Scenario": ""},
#     color_discrete_map={"NetZero": "green", "LeastCost": "aqua"}  
# )

# fig_capital_cost.update_layout(
#     template="plotly_white",
#     # yaxis=dict(range=[0, 20]),  # according to your data range, or let Plotly decide
#     width=width,
#     height=height
# )


# fig.update_layout(template="plotly_white")
# # fig_area1.show()
# fig_area2.show()
# # fig_area3.show()
# # fig_area4.show()
# # fig_stranded_cost.show()
# # fig_needs_by_source.show()
# # fig_elec_cost.show()
# # fig_capital_cost.show()
# # df_long_nz.head(300)


### Here we save the figures 
- In both HTML and .png

### Example usage for the functions:
>
```python
# Import the figure saving utilities
from save_figure import save_figure, save_all_figures

# Basic usage - single line per figure
save_figure(fig_area1)  # Uses figure title as filename
save_figure(fig_area2, custom_name="investment_by_tech") 
save_figure(fig_area3, save_png=False)  # HTML only

# Save to a different directory
save_figure(fig_elec_cost, output_dir="important_figures")

# Save all figures at once
figs = {"area1": fig_area1, "area2": fig_area2, "needs": fig_needs_by_source}
save_all_figures(figs)  # Uses dict keys as filenames

#or 
figs = [fig_area1, fig_area2, fig_needs_by_source]
save_all_figures(figs)  # Uses figure titles as filenames
``` 


In [211]:
# # Create figures directory if it doesn't exist
# import os
# # os.makedirs("figures", exist_ok=True)
# from MinFin.save_figure import save_figure, save_all_figures

# # Save all figures at once (as a list)
# all_figures = [fig_area1, fig_area2, fig_area3, fig_area4, 
#                fig_stranded_cost, fig_needs_by_source, 
#                fig_elec_cost, fig_capital_cost]
# save_all_figures(all_figures)


### Funding Availability Projections

This section analyzes historical and projected funding sources for energy investments:

1. **Data Preparation**:
   - Extracts yearly funding data from `df_funding_envelope`
   - Creates a structured DataFrame with three key funding sources

2. **Visualization**:
   - Creates a bar chart showing historical funding sources by year
   - Generates a stacked area chart displaying future funding projections
   - Uses custom colors to differentiate between funding sources

3. **Output**:
   - Both visualizations help identify funding trends and potential gaps
   - Automatically saves figures to the `figures` directory for later use

The `save_all_figures()` function at the end preserves both visualizations as interactive HTML and static PNG files. 


In [212]:
# import plotly.express as px
# year_idx = [x for x in df_funding_envelope.index
#  if isinstance(x, (int, float))]
# df_funding_historic_plot = df_funding_envelope.loc[year_idx]

# df_funding = pd.DataFrame({
#     "Year": year_idx,  
#     "Government Spending": list(df_funding_historic_plot["Budget"]),
#     "SOE internal cash generation":  list(df_funding_historic_plot["SOE Gen."]),
#     "International Grants":  list(df_funding_historic_plot["Grant"])
# })
# fig = px.bar(
#     df_funding,
#     x="Year",
#     y=["Government Spending", "SOE internal cash generation", "International Grants"],
#     barmode="relative",  
#     title="Historic Sources of Funding Availability for Energy Investments",
#     labels={"value": "Million US$", "variable": ""},  
#     template="plotly_white",
#     color_discrete_sequence=["#fcfab2","#c4b97a", "#d7ea64" ]
# )

# fig.update_layout(
#     xaxis=dict(dtick=1),  
#     yaxis=dict(range=[0, 800]),  
#     width=width,
#     height=height,
#     legend=dict(
#     orientation="h",   
#     yanchor="bottom",
#     y=-0.2,            
#     xanchor="center",
#     x=0.5             
#     )
# )

# fig.show()
# fig_funding_projection = stacked_area_fig(df_funding_plot, stacked_cols_funding,width=800,height=600,color_map=color_map,title=title,line_plots=0)
# fig_funding_projection.show()
# save_all_figures([fig,fig_funding_projection])

### Historic Financing (2010-2024)
This section analyzes the composition of historical energy investment financing from 2010-2024, breaking down funding sources across multiple dimensions:

1 Organization:
> Provider: Public vs. Private sector contributions
> 
> Instrument: Debt vs. Equity financing proportions
> 
> Origin: Domestic vs. International funding sources
> 
> Source: Detailed breakdown by specific financing entities

2 Data:
> Extracts relevant data from financing datasets
> 
> Combines different categories into a unified structure
> 
> Converts wide-format data to long-format for visualization 


In [213]:
# # institution_shares = fbs.get_institution_shares()
# # financing_sector_shares = fbs.get_financing_sector_shares()
# # repayment_statistics
# hd.get_financing_source_shares()

In [214]:
df_provider = financing_sector_shares.loc[["Public", "Private"],:].T
df_instrument = repayment_statistics.loc[("Summary","Total financing volumes"),["Debt Share", "Equity Share"]].T
df_source =hd.get_financing_source_shares().loc[:,"Historical"].to_frame().T
df_source

NameError: name 'hd' is not defined

 In the following cell we:

- Build horizontal stacked bar charts for percentage distributions.
- Plot each financing dimension as its own category.
- Use custom colors for subcategories. 


In [ ]:
import pandas as pd

df_provider = financing_sector_shares.loc[["Public", "Private"],:].T
df_instrument = repayment_statistics.loc[("Summary","Total financing volumes"),["Debt Share", "Equity Share"]].to_frame().T
df_origin = financing_sector_shares.loc[["Domestic", "International"],:].T
df_source = hd.get_financing_source_shares().loc[:,"Historical"].to_frame().T

df_provider["Category"] = "Provider"
df_instrument["Category"] = "Instrument"
df_origin["Category"] = "Origin"
df_source["Category"] = "Source"

# Merge 4 dataframes
df_wide = pd.concat([df_provider, df_instrument, df_origin, df_source], ignore_index=True)

df_long = df_wide.melt(
    id_vars="Category",
    var_name="Subcategory",
    value_name="Value"
)

# If some rows have NaN (e.g. because they don't exist in the original data), discard them
df_long = df_long.dropna(subset=["Value"]).reset_index(drop=True)

import plotly.express as px
# color_map = {
#     "Conc IFI": "#99CCFF",
#     "Conc DPS": "#FFCC99",
#     "Comm Intl": "#FF9999",
#     "Comm Dom": "#FFFF99",
#     "Domestic": "#FFCC66",
#     "International": "#66CC99",
#     "Debt": "#99FF99",
#     "Equity": "#66CC66",
#     "Public": "#CCCCFF",
#     "Private": "#CC99FF"
# }

fig = px.bar(
    df_long,
    x="Value",
    y="Category",
    color="Subcategory",
    text="Subcategory",
    orientation="h",
    barmode="relative",  # stack
    color_discrete_map=color_map,
    title="Historic Sources of Finance for Energy Investments 2010-2024",
    labels={"Value": "Percentage (%)", "Category": ""}
)
fig.update_traces(
    # ,      
    textposition="inside",    # put text inside the bar
    insidetextanchor="middle" # center the text vertically
)

fig.update_xaxes(range=[0, 1])

fig.update_layout(
    template="plotly_white",
    showlegend = False,
    width = width,
    height = height,
    # margin=dict(l=0, r=0, t=50, b=0)
)

fig.show()
save_figure(fig)

Saved HTML: minfin_output/figures\Historic_Sources_of_Finance_for_Energy_Investments_2010-2024.html
Error saving PNG: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



{'html': 'minfin_output/figures\\Historic_Sources_of_Finance_for_Energy_Investments_2010-2024.html'}

In [ ]:
repayment_statistics

Volume (USD) Interest rate       Term  \
Summary Total financing volumes  10983.174661      0.053349  27.139904   
        Conc_IFI                  8169.432921      0.024346  27.211158   
        Conc_DPS                   1220.21174      0.168237   32.11396   
        Comm_Intl                    1337.892      0.108661  22.029274   
        Comm_Dom                      255.638      0.142344  27.867336   
Equity  Total financing volumes   2207.860073      0.157316  29.849349   
        Conc_IFI                   135.518333       0.15093    24.8871   
        Conc_DPS                   1014.23174       0.19702  34.037672   
        Comm_Intl                     853.872      0.115266  25.574665   
        Comm_Dom                      204.238      0.140188  30.214505   
Debt    Total financing volumes   8775.314587      0.027191   26.45821   
        Conc_IFI                  8033.914587      0.022211  27.250361   
        Conc_DPS                       205.98      0.026511  22.641732   
        Comm_Intl                      484.02      0.097009  15.774761   
        Comm_Dom                         51.4      0.150913  18.540856   

                                Grace period Average Annual Payment  \
Summary Total financing volumes     5.355647              10.020328   
        Conc_IFI                    6.807053                8.83174   
        Conc_DPS                    1.198324              21.692998   
        Comm_Intl                   1.019163               7.083187   
        Comm_Dom                    1.511904               7.659716   
Equity  Total financing volumes          0.0              15.084025   
        Conc_IFI                         0.0               4.483015   
        Conc_DPS                         0.0              25.284345   
        Comm_Intl                        0.0               6.290694   
        Comm_Dom                         0.0               8.226935   
Debt    Total financing volumes     6.703122               8.746307   
        Conc_IFI                    6.921876               8.905095   
        Conc_DPS                    7.098791               4.009441   
        Comm_Intl                   2.817094               8.481244   
        Comm_Dom                    7.519455               5.405874   

                                 Debt Share  Equity Share  Market Element  \
Summary Total financing volumes    0.798978      0.201022             NaN   
        Conc_IFI                   0.983412      0.016588             NaN   
        Conc_DPS                   0.168807      0.831193             NaN   
        Comm_Intl                  0.361778      0.638222             NaN   
        Comm_Dom                   0.201066      0.798934             NaN   
Equity  Total financing volumes         NaN           NaN             NaN   
        Conc_IFI                        NaN           NaN             NaN   
        Conc_DPS                        NaN           NaN             NaN   
        Comm_Intl                       NaN           NaN             NaN   
        Comm_Dom                        NaN           NaN             NaN   
Debt    Total financing volumes         NaN           NaN        0.220326   
        Conc_IFI                        NaN           NaN        0.193863   
        Conc_DPS                        NaN           NaN        0.228687   
        Comm_Intl                       NaN           NaN        0.661032   
        Comm_Dom                        NaN           NaN        0.173028   

                                 Grant Element  
Summary Total financing volumes            NaN  
        Conc_IFI                           NaN  
        Conc_DPS                           NaN  
        Comm_Intl                          NaN  
        Comm_Dom                           NaN  
Equity  Total financing volumes            NaN  
        Conc_IFI                           NaN  
        Conc_DPS                           NaN  
        Comm_Intl                    

In [ ]:
import plotly.graph_objects as go
import pandas as pd
volume={"Conc_IFI": 0,
    "Conc_DPS":  0,
    "Comm_Intl":  0,
    "Comm_Dom":   0}
df_raw = pd.DataFrame({
    "MarketElement": list(repayment_statistics.loc[("Debt",list(volume.keys())),"Market Element"]),
    "volume": list(repayment_statistics.loc[("Debt",list(volume.keys())),"Volume (USD)"]),
    "color": ["#ffbbbc", "#ff7b7b", "#ff0000", "#c00000"]
},index=volume.keys())

df_sorted = df_raw.sort_values("MarketElement")
df_plot = df_sorted.copy()


base = df_raw.loc[:,"volume"].cumsum()-df_raw.loc[:,"volume"]

fig = go.Figure()

a = df_plot.loc[:,"volume"].cumsum()
for key, v in volume.items():
    y_val = df_plot.loc[key,"MarketElement"]
    x_val = df_plot.loc[key,"volume"]#+base[key]
    # print([y_val])
    fig.add_trace(
        go.Bar(
            x=[x_val],         
            y=[y_val],
            base= base[key],
            orientation="h",
            marker_color=df_plot.loc[key,"color"],
            name=key,
            width=y_val,              
            offset=-y_val              # offset 0.15, align the down edge of the bar to y_val
        )
    )

fig.update_layout(
    title="Merit order of Financing Sources by Grant Element between 2010-2024",
    showlegend=True,
    barmode="relative",   
    xaxis=dict(title="Amount ($)", range=[0, 9000]),
    yaxis=dict(title="Market Element", range=[0,1], zeroline=True),
    template="plotly_white",
    width=width,
    height=height
)
save_figure(fig)
fig.show()


Saved HTML: minfin_output/figures\Merit_order_of_Financing_Sources_by_Grant_Element_between_2010-2024.html
Error saving PNG: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



 >Potential issue of the above fig 


In [ ]:
df_plot

,MarketElement,volume,color
Comm_Dom,0.173028,51.400000,#c00000
Conc_IFI,0.193863,8033.914587,#ffbbbc
Conc_DPS,0.228687,205.980000,#ff7b7b
Comm_Intl,0.661032,484.020000,#ff0000


### Financial Institution Contributions Analysis (2010-2024)

This section visualizes the proportional contributions from different financial institutions to energy investments during 2010-2024:

1. **Data Preparation**:
   - Organizes institution contribution data into plotting format
   - Adds empty placeholder column for proper horizontal display

2. **Custom Visualization Function**:
   - Implements a reusable `single_bar_plot()` function for horizontal stacked bars
   - Handles percentage formatting when values are between 0-1
   - Positions the legend horizontally at the bottom for better readability

3. **Output**:
   - Displays the relative contribution of each financial institution
   - Shows the historic proportion of funding from each source
   - Helps identify the most significant financial contributors

The visualization provides insights into which institutions have historically played the largest roles in financing energy investments. 


In [ ]:
import plotly.express as px
from MinFin.save_figure import save_figure
from MinFin.plotting_notebook import single_bar_plot

color_map = {
    "Bilateral Agency": "#7f6000",
    "Multilateral Agency": "#bf9000",
    "Foreign Government": "#ffc000",
    "National Government": "#ffd966",
    "Domestic Public Sector": "#ffe699",
    "Climate Funds": "#fff2cc",
    "Commercial Bank": "#2f5597",
    "Private Equity Fund": "#4472c4",
}
df_institution_shares_plot = pd.DataFrame(
    {"Financier": institution_shares.index, "Share": institution_shares["Share"]}
)
df_institution_shares_plot[""] = ""

fig = single_bar_plot(
    df_institution_shares_plot,
    "Share",
    "",
    color_map,
    "Historic Proportion of Contributions from Financier 2010-2024",
    {"Share": "Percentage", "Row": ""},
)
save_figure(fig)
fig.show()


Saved HTML: minfin_output/figures\Historic_Proportion_of_Contributions_from_Financier_2010-2024.html
Error saving PNG: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



### Historical Financing Sources Visualisation

This section analyzes the breakdown of financing by source type and financial instrument (debt vs. equity; Domestic vs. international):

1. **Data Extraction**:
   - Identifies four key financing source types: Concessional IFI, Concessional DPS, Commercial International, and Commercial Domestic
   - Extracts volume data for both debt and equity instruments across these sources
   - Combines source type and instrument into unified categories (e.g., "Conc_IFI (Debt)")

2. **Visualization Preparation**:
   - Creates a custom color scheme to distinguish between different financing sources
   - Calculates percentage shares for relative comparison
   - Prepares data for horizontal bar chart visualization

3. **Multiple Views**:
   - First chart shows the relative share (percentage) of each financing source-instrument combination
   - Second chart presents the absolute volumes in USD
   - Both visualizations use the same color scheme for consistency 


In [ ]:
repayment_statistics
types = ["Conc_IFI", "Conc_DPS", "Comm_Intl", "Comm_Dom"]
df_new = repayment_statistics.loc[(["Debt","Equity"],types),"Volume (USD)"]
# df_new = df_new.rename(columns={"value": "Amount"})

In [ ]:
types = ["Conc_IFI", "Conc_DPS", "Comm_Intl", "Comm_Dom"]
df_source_share = repayment_statistics.loc[(["Debt","Equity"],types),"Volume (USD)"]

df_source_share = df_source_share.reset_index() # reset multi-level index to single level
#  ["level_0", "level_1", "value"]

df_source_share["merged"] = df_source_share["level_1"] + " (" + df_source_share["level_0"] + ")"

color_map_source_share = ["#7f6000", "#bf9000", "#ffc000","#ffd966","#ffe699","#fff2cc","#2f5597","#4472c4"]
color_map_source_share = {k:v for k,v in zip(df_source_share["merged"].unique(), color_map_source_share)}

df_source_share_plot = pd.DataFrame({
    "Financier": df_source_share["merged"],
    "Share": df_source_share["Volume (USD)"]/df_source_share["Volume (USD)"].sum()
})
df_source_share_plot[""] = ""
fig = single_bar_plot(df_source_share_plot, "Share", "", color_map_source_share, "Historic Proportion of Contributions from Different Sources 2010-2024", {"Share": "Percentage", "Row": ""})

save_figure(fig)
fig.show()

df_source_share_plot = pd.DataFrame({
    "Financier": df_source_share["merged"],
    "Share": df_source_share["Volume (USD)"]
})
df_source_share_plot[""] = ""
fig = single_bar_plot(df_source_share_plot, "Share", "", color_map_source_share,  "Historic Proportion of Contributions from Different Sources 2010-2024", {"Share": "Percentage", "Row": ""})

save_figure(fig)
fig.show()


Saved HTML: minfin_output/figures\Historic_Proportion_of_Contributions_from_Different_Sources_2010-2024.html
Error saving PNG: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



Saved HTML: minfin_output/figures\Historic_Proportion_of_Contributions_from_Different_Sources_2010-2024.html
Error saving PNG: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



In [ ]:
institution_shares

,Share,Market Element
Bilateral Agency,0.0,NaN
Multilateral Agency,0.0,NaN
Foreign Government,0.0,NaN
National Government,0.0,NaN
Domestic Public Sector,0.0,NaN
Climate Funds,0.0,NaN
Commercial Bank,0.0,NaN
Private Equity Fund,0.0,NaN


In [ ]:
df_category_sum_nz.reset_index()

Technology,index,Year,BESS_TECH,Distribution Infrastructure,Generation Fossil-Fuel,Generation Renewable,Transmission Infrastructure,Year,BESS_TECH,Biomass,...,Oil,Solar PV,Transmission,Wind,CSP,Coal,Energy Exports,Direct Solar,Direct Oil,Direct Hydro
0,0,2025.0,103.2542,191.7485,0.0000,298.3534,29.2420,2025.0,103.2542,0.00,...,0.0,0.0000,29.2420,19.0608,0.0,0.0,0.0,0.0,0.0,0.0
1,1,2026.0,0.0000,168.7515,0.0000,783.6879,25.7349,2026.0,0.0000,25.00,...,0.0,41.1540,25.7349,86.8715,0.0,0.0,0.0,0.0,0.0,0.0
2,2,2027.0,0.0000,318.4243,83.0037,516.7908,48.5603,2027.0,0.0000,0.00,...,0.0,0.0000,48.5603,74.4683,0.0,0.0,0.0,0.0,0.0,0.0
3,3,2028.0,0.0000,348.2791,19.8531,533.1626,53.1132,2028.0,0.0000,20.00,...,0.0,0.0000,53.1132,170.5260,0.0,0.0,0.0,0.0,0.0,0.0
4,4,2029.0,0.0000,393.2781,0.0000,1184.0274,59.9757,2029.0,0.0000,0.00,...,0.0,0.0000,59.9757,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
5,5,2030.0,0.0000,384.6090,58.2855,891.4553,58.6536,2030.0,0.0000,120.50,...,0.0,0.0000,58.6536,426.0647,0.0,0.0,0.0,0.0,0.0,0.0
6,6,2031.0,162.2502,367.2810,0.0000,677.4076,56.0110,2031.0,162.2502,50.00,...,0.0,0.0000,56.0110,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
7,7,2032.0,33.7460,424.6634,0.0000,745.8637,64.7620,2032.0,33.7460,27.00,...,0.0,0.0000,64.7620,442.6991,0.0,0.0,0.0,0.0,0.0,0.0
8,8,2033.0,0.0000,412.6597,0.0000,879.8064,62.9314,2033.0,0.0000,7.50,...,0.0,0.0000,62.9314,596.6576,0.0,0.0,0.0,0.0,0.0,0.0
9,9,2034.0,0.0000,451.1339,0.0000,953.3591,68.7988,2034.0,0.0000,15.00,...,0.0,0.0000,68.7988,0.0000,0.0,0.0,0.0,0.0,0.0,0.0


### Market element visualisation 


In [ ]:
from MinFin.plotting_notebook import plot_market_element_by_financier
from MinFin.save_figure import save_figure

color_map = {
    "Bilateral Agency": "#7f6000",
    "Multilateral Agency": "#bf9000",
    "Foreign Government": "#ffc000",
    "National Government": "#ffd966",
    "Domestic Public Sector": "#ffe699",
    "Climate Funds": "#fff2cc",
    "Commercial Bank": "#2f5597",
    "Private Equity Fund": "#4472c4",
}

fig = plot_market_element_by_financier(
    institution_shares["Market Element"].to_frame(),
    color_map=color_map,
)
save_figure(fig)
fig


Saved HTML: minfin_output/figures\Market_Elements_of_Finance_for_Energy_Investments_2010-2024_by_Financier.html
Error saving PNG: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



### Techonlogy level financing stats  


In [ ]:
fbs.get_technology_stats()

,IRR,Volume of Finance,Interest Rate,Debt Share,Equity Share,WACC
Technology,,,,,,
Geothermal,0.200115,4410.840921,0.035600,0.756456,0.243544,0.075667
Solar PV,0.050575,783.025000,0.037162,0.747792,0.252208,0.040545
Biomass,0.120000,229.000000,0.061521,0.615721,0.384279,0.083993
Transmission,0.132620,1977.370902,0.023646,0.878075,0.121925,0.036933
Gas,0.000000,4.300000,0.040960,1.000000,0.000000,0.040960
Oil,0.169600,408.300000,0.076433,0.743938,0.256062,0.100290
Distribution,0.092178,1769.357838,0.023167,0.973866,0.026134,0.024971
Onshore Wind,0.153333,1165.380000,0.032506,0.799585,0.200415,0.056722
Hydropower,0.053000,85.600000,0.052250,0.642523,0.357477,0.052518


In [ ]:
import pandas as pd
import plotly.express as px

# get the technology stats, use the function from fbs(financing baseline stats)

df_technology_financing = fbs.get_technology_stats()
df_technology_financing["Technology"] = df_technology_financing.index

# convert the data to long format(melt)
df_melted = df_technology_financing.melt(id_vars=["Technology"], value_vars=["IRR", "Interest Rate", "WACC"],
                     var_name="Metric", value_name="Value")

color_map = {"IRR": "#00ffcc", "Interest Rate": "#ffc000", "WACC": "#ff99ff"}    

fig = px.bar(
    df_melted,
    x="Technology",
    y="Value",
    color="Metric",
    title="Historic Cost of Financing 2010-2024",
    barmode="group",
    color_discrete_map=color_map,
    labels={"Value": "Percentage", "Technology": "Technology"},
    
)
fig.update_traces(marker=dict(line=dict(color='black', width=2)))

# adjust the Y axis to be percentage format
fig.update_layout(
    yaxis_tickformat=".0%",
    width=width,
    height=height,
    )

save_figure(fig)
fig.show()


Saved HTML: minfin_output/figures\Historic_Cost_of_Financing_2010-2024.html
Error saving PNG: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



### Debt Equity financing stats for energy investments 


In [ ]:
#get data required for the plot
terms_debt_finance = repayment_statistics.loc["Debt",:]
terms_debt_finance
# Weighted average not used in EXCEL

,Volume (USD),Interest rate,Term,Grace period,Average Annual Payment,Debt Share,Equity Share,Market Element,Grant Element
Total financing volumes,8775.314587,0.027191,26.45821,6.703122,8.746307,NaN,NaN,0.220326,0.779674
Conc_IFI,8033.914587,0.022211,27.250361,6.921876,8.905095,NaN,NaN,0.193863,0.806137
Conc_DPS,205.98,0.026511,22.641732,7.098791,4.009441,NaN,NaN,0.228687,0.771313
Comm_Intl,484.02,0.097009,15.774761,2.817094,8.481244,NaN,NaN,0.661032,0.338968
Comm_Dom,51.4,0.150913,18.540856,7.519455,5.405874,NaN,NaN,0.173028,0.826972


In [ ]:
institution_shares

,Share,Market Element
Bilateral Agency,0.0,NaN
Multilateral Agency,0.0,NaN
Foreign Government,0.0,NaN
National Government,0.0,NaN
Domestic Public Sector,0.0,NaN
Climate Funds,0.0,NaN
Commercial Bank,0.0,NaN
Private Equity Fund,0.0,NaN


 - By Financier 


In [ ]:
# import pandas as pd
# from MinFin.plotting_notebook import debt_equity_plot
# from MinFin.save_figure import save_figure

# institution_shares.sort_values("Equity Share", inplace=True)
# data = {
#     "Financier": institution_shares.index,
#     "Debt Share": institution_shares["Debt Share"],
#     "Equity Share": institution_shares["Equity Share"],
# }
# df_debt_eq = pd.DataFrame(data)
# fig = debt_equity_plot(df_debt_eq, "Debt Equity in Financing for Energy Investments 2010-2024 by Financier")
# save_figure(fig)
# fig


 - By Source 


In [ ]:
repayment_statistics.loc["Summary",["Debt Share","Equity Share"]]

,Debt Share,Equity Share
Total financing volumes,0.798978,0.201022
Conc_IFI,0.983412,0.016588
Conc_DPS,0.168807,0.831193
Comm_Intl,0.361778,0.638222
Comm_Dom,0.201066,0.798934


In [ ]:
filtered_data = repayment_statistics.loc["Summary",["Debt Share","Equity Share"]].copy()
filtered_data.sort_values("Equity Share", inplace=True)
filtered_data = filtered_data.loc[filtered_data.index != "Total financing volumes",:]
data = {
    "Financier": filtered_data.index,
    "Debt Share": filtered_data["Debt Share"],
    "Equity Share": filtered_data["Equity Share"]
}
df = pd.DataFrame(data)    
fig=debt_equity_plot(df, "Debt Equity in Financing for Energy Investments 2010-2024 by Source")
save_figure(fig)
fig


Saved HTML: minfin_output/figures\Debt_Equity_in_Financing_for_Energy_Investments_2010-2024_by_Source.html
Error saving PNG: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



 - By Technology 


In [ ]:
technology_debt_equity_share = fbs.get_technology_stats()[["Debt Share", "Equity Share"]]
technology_debt_equity_share.sort_values("Equity Share", inplace=True)
data = {
    "Financier": technology_debt_equity_share.index,
    "Debt Share": technology_debt_equity_share["Debt Share"],
    "Equity Share": technology_debt_equity_share["Equity Share"]
}
df = pd.DataFrame(data)  
fig = debt_equity_plot(df, "Debt Equity in Financing for Energy Investments 2010-2024 by Source")
save_figure(fig)
fig

Saved HTML: minfin_output/figures\Debt_Equity_in_Financing_for_Energy_Investments_2010-2024_by_Source.html
Error saving PNG: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



### Future Financing Needs Projections

This section forecasts the projected financing requirements broken down by source type and instrument:

1. **Data Visualization**:
   - Presents a stacked area chart showing projected financing needs over time (2025-2050)
   - Categorizes financing by both source (IFI, DPS, International Commercial, Domestic Commercial) and instrument type (Debt vs. Equity)

2. **Financing Categories**:
   - **Conc IFI (Debt/Equity)**: Concessional financing from International Financial Institutions
   - **Conc DPS (Debt/Equity)**: Concessional financing from Domestic Public Sector
   - **Comm Intl (Debt/Equity)**: Commercial financing from International sources
   - **Comm Dom (Debt/Equity)**: Commercial financing from Domestic sources

3. **Trend Analysis**:
   - Illustrates how financing needs evolve over the projection period
   - Shows the changing composition of financing sources and instruments
   - Highlights periods of peak financing requirements

- Repayment projection  


In [ ]:
# years = list(nz_needs.columns)
# df_stack_nz_needs = pd.DataFrame({
#     "Year": years,
#     "Conc IFI (Debt)": list(nz_needs.loc["Debt: Conc_IFI", years]),
#     "Conc DPS (Debt)": list(nz_needs.loc["Debt: Conc_DPS", years]),
#     "Comm Intl (Debt)": list(nz_needs.loc["Debt: Comm_Intl", years]),
#     "Comm Dom (Debt)": list(nz_needs.loc["Debt: Comm_Dom", years]),
#     "Conc IFI (Equity)": list(nz_needs.loc["Equity: Conc_IFI", years]),
#     "Conc DPS (Equity)": list(nz_needs.loc["Equity: Conc_DPS", years]),
#     "Comm Intl (Equity)": list(nz_needs.loc["Equity: Comm_Intl", years]),
#     "Comm Dom (Equity)": list(nz_needs.loc["Equity: Comm_Dom", years])
# })

# stacked_cols=list(df_stack_nz_needs.columns[1:])
# #["Debt: Conc_IFI", "Debt: Conc_DPS", "Debt: Comm_Intl", "Debt: Comm_Dom", 
# #"Equity: Conc_IFI", "Equity: Conc_DPS", "Equity: Comm_Intl", "Equity: Comm_Dom"]

# color_map = {
#     "Conc IFI (Debt)": "#7f6000",
#     "Conc DPS (Debt)": "#bf9000",
#     "Comm Intl (Debt)": "#ffc000",
#     "Comm Dom (Debt)": "#ffe699",
#     "Conc IFI (Equity)": "#203864",
#     "Conc DPS (Equity)": "#2f5597",
#     "Comm Intl (Equity)": "#4472c4",
#     "Comm Dom (Equity)": "#b4c7e7" 
# }
# colors = list(color_map.values())
# color_map_nz_repay = {k: list(color_map.values())[i] for i,k in enumerate(stacked_cols) }

# f = stacked_area_fig(
#     df_stack_nz_needs, 
#     stacked_cols,
#     width=800,height=600,
#     color_map=color_map_nz_repay,
#     title="Net Zero Financing Repayment by Source",
#     line_plots=False)
# save_figure(f)
# f

In [ ]:
# nz_needs = hd.get_net_zero_financing_needs_full(df_invest_need_summary,df_funding_envelope) # Net Zero financing needs
# lc_needs = hd.get_least_cost_financing_needs_full(df_invest_need_summary,df_funding_envelope) # Least Cost financing needs
# additional_investment_needs = hd.get_additional_investment_needs(lc_needs,nz_needs) #Additional needs (NetZero - LeastCost)

# needs_of_different_scenario = {"NetZero":nz_needs,"LeastCost":lc_needs,"Incremental":additional_investment_needs} #put them in one dict for further processing 


# df_financing_repayments= hd.get_repayments(repayment_schedule,df_invest_need_summary,df_funding_envelope)
# df_financing_repayments



In [ ]:

# years = list(df_financing_repayments.columns)
# df_stack_future_repayment = pd.DataFrame({
#     "Year": years,
#     "Existing finance payments": list(df_financing_repayments.loc["Existing finance payments (Million USD)", years]),
#     "Construction of new energy": list(df_financing_repayments.loc["Financing payments for construction of new energy", years]),
# })
# stacked_cols=list(df_stack_future_repayment.columns[1:])

# color_map = {
#     "Financing Payments for retirement of old fossil fuel technologies": "#ff0003",
#     "Existing finance payments": "#ff9999",
#     "Financing payments for construction of new energy": "#a5a5a5",
# }
# colors = list(color_map.values())
# color_map_future_repayment = {k: list(color_map.values())[i] for i,k in enumerate(stacked_cols) }

# fig_future_repayment = stacked_area_fig(
#     df_stack_future_repayment, 
#     stacked_cols,
#     width=800,height=600,
#     color_map=color_map_future_repayment,
#     title="Net Zero Financing Repayment by Source",
#     line_plots=False)
# save_figure(fig_future_repayment)
# fig_future_repayment

NameError: name 'df_financing_repayments' is not defined

In [ ]:
# import numpy as np
# import pandas as pd
# import plotly.graph_objects as go

# # --- Parameters ---
# YEAR_START, YEAR_END = 2025, 2050
# years = list(range(YEAR_START, YEAR_END + 1))

# # Annual funding availability (must match dashboard_summary after .T; row name may need adjusting)
# fa = dashboard_summary.loc["funding_availability"].reindex(years).fillna(0)

# # --- Financing requirement by technology and year ---
# def _req_series(tech):
#     s = financing_requirement_by_tech[tech].loc["Financing Requirement"]
#     return s.reindex(years).fillna(0)

# techs = list(financing_requirement_by_tech.keys())
# req_df = pd.DataFrame({t: _req_series(t) for t in techs}).T  # rows=tech, cols=year

# # --- Allocate total funding availability by share of yearly financing requirement ---
# total_req_y = req_df.sum(axis=0)
# with np.errstate(divide="ignore", invalid="ignore"):
#     share = req_df.div(total_req_y.replace(0, np.nan), axis=1).fillna(0.0)
# alloc_df = share.mul(fa, axis=1)

# gap_df = req_df - alloc_df
# gap_by_tech = gap_df.sum(axis=1).sort_values(ascending=False)

# # --- Sanity check: sum of tech gaps per year should match dashboard funding_shortfall ---
# check = gap_df.sum(axis=0)
# ref = (
#     dashboard_summary.loc["financing_requirement"].reindex(years).fillna(0)
#     - fa
# )
# print("Max abs diff (per-year total vs funding_shortfall):", (check - ref).abs().max())

# # --- Plot (plotly_white) ---
# width, height = 800, 600

# color_map = {
#     "Biomass": "#ff9900",
#     "Geothermal": "#66ccff",
#     "Solar PV": "#ffd700",
#     "Wind": "#66ff66",
#     "Onshore Wind": "#66ff66",
#     "Offshore Wind": "#66ff66",
#     "Nuclear": "#999999",
#     "Coal": "#333333",
#     "Gas": "#ff6666",
#     "Oil": "#cc6600",
#     "Hydropower": "#3399FF",
#     "CSP": "#FFB84D",
#     "Distribution": "#808080",
#     "Transmission": "#9966FF",
#     "Battery": "#19d3f3",
#     "Imports": "#2f5597",
#     "Other": "#BEBEBE",
# }
# bar_colors = [color_map.get(t, "#888888") for t in gap_by_tech.index]

# fig = go.Figure(
#     go.Bar(
#         x=gap_by_tech.index.astype(str),
#         y=gap_by_tech.values,
#         marker=dict(color=bar_colors, line=dict(color="black", width=1)),
#         hovertemplate="%{x}<br>Gap (Million US$)=%{y:,.0f}<extra></extra>",
#     )
# )
# fig.update_layout(
#     title=f"Finance Gap by Technology ({YEAR_START}–{YEAR_END}) — requirement vs allocated funding",
#     template="plotly_white",
#     width=width,
#     height=height,
#     xaxis=dict(tickangle=-45, title="Technology"),
#     yaxis=dict(title="Million US$", showgrid=True, gridcolor="lightgray", zeroline=True),
#     showlegend=False,
# )

# try:
#     from MinFin.save_figure import save_figure
#     save_figure(fig, custom_name="Finance_Gap_by_Technology_requirement_vs_funding")
# except Exception as e:
#     print("save_figure:", e)

# fig.show()


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from MinFin.plotting_notebook import plot_technology_cashflow_waterfall
from MinFin.save_figure import save_figure

TECH = "Biomass"
YEAR = 2035

fig = plot_technology_cashflow_waterfall(
    TECH,
    YEAR,
    tech_dataframes,
    financing_requirement_by_tech,
    existing_financing_repayment=None,
)

try:
    save_figure(fig, custom_name=f"Technology_Cashflow_{TECH}_{YEAR}")
except Exception as e:
    print("save_figure:", e)

fig.show()


Saved HTML: minfin_output/figures\Technology_Cashflow_Biomass_2035.html
Error saving PNG: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



In [ ]:
# from MinFin.market_revenue import (
#     aggregate_market_revenue_stacks,
#     plot_net_zero_funding_sources_figure,
# )

# years = list(range(2025, 2051))

# ppa, enduser, wholesale = aggregate_market_revenue_stacks(
#     tech_dataframes,
#     years,
#     exchange_rates,
# )

# fr = dashboard_summary.loc["financing_requirement"].reindex(years).fillna(0)

# fig = plot_net_zero_funding_sources_figure(
#     years,
#     ppa,
#     enduser,
#     wholesale,
#     fr,
# )
# fig.show()


In [ ]:
# 